In [ ]:
# Cell 1 — Environment diagnostic
import os, subprocess
from google.cloud import bigquery, storage

PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT') or subprocess.check_output(
    ['gcloud', 'config', 'get-value', 'project']
).decode().strip()
ACCOUNT = subprocess.check_output(
    ['gcloud', 'config', 'get-value', 'account']
).decode().strip()

print(f"Project: {PROJECT_ID}")
print(f"Account: {ACCOUNT}")

# --- BigQuery check ---
bq = bigquery.Client(project=PROJECT_ID)
print("\n--- BigQuery: fe_race10 ---")
try:
    ds = bq.get_dataset(f"{PROJECT_ID}.fe_race10")
    print(f"Dataset location: {ds.location}")
    tables = list(bq.list_tables(ds))
    print(f"Tables ({len(tables)}):")
    for t in tables:
        ft = bq.get_table(t.reference)
        print(f"  {t.table_id:35s} {ft.num_rows:>12,} rows  {ft.num_bytes/1e6:>8.1f} MB")
except Exception as e:
    print(f"NOT FOUND in this project: {type(e).__name__}: {e}")

# --- GCS check ---
print("\n--- GCS: class-demo/formula-e/r10/ ---")
gcs = storage.Client(project=PROJECT_ID)
try:
    blobs = list(gcs.list_blobs('class-demo', prefix='formula-e/r10/', max_results=30))
    print(f"Reachable. First {len(blobs)} entries:")
    for b in blobs:
        print(f"  {b.name:70s} {b.size/1e6:>8.1f} MB")
except Exception as e:
    print(f"NOT REACHABLE: {type(e).__name__}: {e}")

# --- Pub/Sub permissions check ---
print("\n--- Pub/Sub: topic existence ---")
from google.cloud import pubsub_v1
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(PROJECT_ID, 'fe-telemetry')
try:
    topic = publisher.get_topic(request={"topic": topic_path})
    print(f"Topic exists: {topic.name}")
except Exception as e:
    print(f"Topic does NOT exist (we'll create it later): {type(e).__name__}")

Project: qwiklabs-gcp-03-ec1498964e29
Account: student-03-c4e40b20ffae@qwiklabs.net

--- BigQuery: fe_race10 ---
Dataset location: us-central1
Tables (14):
  attack                                       129 rows       0.0 MB
  career_driver                                 87 rows       0.0 MB
  career_race                                2,799 rows       0.8 MB
  drivers                                       22 rows       0.0 MB
  energy_per_lap                               852 rows       0.1 MB
  event_stream                               1,787 rows       0.2 MB
  laps                                         854 rows       0.3 MB
  racecontrol_classified                        76 rows       0.0 MB
  startgrid                                     22 rows       0.0 MB
  telemetry                              1,281,780 rows     159.2 MB
  top_speed_per_lap                            854 rows       0.0 MB
  v_am_with_driver                               0 rows       0.0 MB
  v_laps_with_dr

In [ ]:
# Cell 2 — Find the real R10 prefix and inventory the files we need
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket('class-demo')

# Walk the formula-e tree a few levels deep to find the real layout
print("=== Top of formula-e/ ===")
for b in gcs.list_blobs('class-demo', prefix='formula-e/', delimiter='/', max_results=50):
    pass  # blobs at this level (probably few)
# delimiter trick: prefixes (folders) come back on the iterator's .prefixes
it = gcs.list_blobs('class-demo', prefix='formula-e/', delimiter='/')
list(it)  # exhaust to populate .prefixes
print("Folders under formula-e/:")
for p in it.prefixes:
    print(f"  {p}")

# Now drill into the most likely R10 path
candidates = [
    'formula-e/berlin_2024/r10/',
    'formula-e/r10/',
    'formula-e/berlin_2024/',
]
for prefix in candidates:
    blobs = list(gcs.list_blobs('class-demo', prefix=prefix, max_results=5))
    print(f"\n{prefix} -> {len(blobs)} blob(s) in first page")
    for b in blobs[:5]:
        print(f"    {b.name}")

# Once we know the real R10 prefix, list the files we actually need for Fork 1
print("\n=== Looking for R10 Parquet files we need ===")
needed = ['energy/energy_per_lap', 'timing/attack', 'timing/laps',
          'timing/timing', 'timing/racecontrol_classified', 'timing/drivers',
          'derived/event_stream']
for needle in needed:
    matches = [b.name for b in gcs.list_blobs('class-demo', prefix='formula-e/')
               if 'r10' in b.name and needle in b.name and b.name.endswith('.parquet')]
    print(f"  {needle:35s} -> {matches[0] if matches else 'NOT FOUND'}")

# Telemetry is partitioned by car_number - count partitions
tele_blobs = [b.name for b in gcs.list_blobs('class-demo', prefix='formula-e/')
              if 'r10' in b.name and 'telemetry' in b.name and b.name.endswith('.parquet')]
print(f"\nR10 telemetry partitions found: {len(tele_blobs)}")
for b in tele_blobs[:3]:
    print(f"  sample: {b}")

=== Top of formula-e/ ===
Folders under formula-e/:
  formula-e/reference/
  formula-e/berlin_2024/
  formula-e/r10/
  formula-e/footage/
  formula-e/cross_challenge/

formula-e/berlin_2024/r10/ -> 5 blob(s) in first page
    formula-e/berlin_2024/r10/derived/event_stream.parquet
    formula-e/berlin_2024/r10/derived/racecontrol_classified.parquet
    formula-e/berlin_2024/r10/energy/energy_per_lap.parquet
    formula-e/berlin_2024/r10/telemetry/car_number=1/022371de34b4438ca51b0c01e39c1bcb-0.parquet
    formula-e/berlin_2024/r10/telemetry/car_number=11/022371de34b4438ca51b0c01e39c1bcb-0.parquet

formula-e/r10/ -> 1 blob(s) in first page
    formula-e/r10/simulator/frames_v1.jsonl.gz

formula-e/berlin_2024/ -> 5 blob(s) in first page
    formula-e/berlin_2024/r09/derived/event_stream.parquet
    formula-e/berlin_2024/r09/derived/racecontrol_classified.parquet
    formula-e/berlin_2024/r09/energy/energy_per_lap.parquet
    formula-e/berlin_2024/r09/telemetry/car_number=1/beb5db8277124e4

In [ ]:
# Cell 3 — Survey the R10 source files
import pandas as pd

R10 = 'gs://class-demo/formula-e/berlin_2024/r10'

def survey(name, path, head_n=3):
    df = pd.read_parquet(path)
    print(f"\n=== {name} ===  ({path.replace('gs://class-demo/formula-e/berlin_2024/r10/', '')})")
    print(f"shape: {df.shape}")
    print(f"dtypes:\n{df.dtypes.to_string()}")
    print(f"head({head_n}):\n{df.head(head_n).to_string()}")
    return df

energy   = survey('ENERGY per lap',  f'{R10}/energy/energy_per_lap.parquet')
attack   = survey('ATTACK events',   f'{R10}/timing/attack.parquet')
laps     = survey('LAPS',            f'{R10}/timing/laps.parquet')
timing   = survey('TIMING events',   f'{R10}/timing/timing.parquet')
rc       = survey('RACE CONTROL',    f'{R10}/derived/racecontrol_classified.parquet')
drivers  = survey('DRIVERS',         f'{R10}/timing/drivers.parquet')
events   = survey('EVENT STREAM',    f'{R10}/derived/event_stream.parquet')

# Key validations
print("\n\n=== KEY CHECKS ===")

# 1. Per-car energy sums to 100 for finishers
print("\n1. Per-car energy sums (should be ~100 for finishers):")
print(energy.groupby('car_number')['percent_consumed'].sum().describe())

# 2. AM scenario column distribution
print(f"\n2. AM scenario values seen: {sorted(attack['scenario'].dropna().unique().tolist())}")
print(f"   AM rows with active=True: {(attack['active']==True).sum()}")
print(f"   AM rows with active=False: {(attack['active']==False).sum()}")

# 3. Race start / end times from laps and race control
print(f"\n3. Race timing:")
print(f"   Earliest lap start_time: {laps['start_time'].min()}")
print(f"   Latest lap start_time:   {laps['start_time'].max()}")
print(f"   RC categories seen: {sorted(rc['category'].unique().tolist())}")
print(f"   First RC message: {rc['time'].min()}")
print(f"   Last RC message:  {rc['time'].max()}")

# 4. Look for green flag / chequered flag specifically
print(f"\n4. Flag messages:")
flag_cats = rc[rc['category'].str.contains('FLAG|START|CHEQUER', case=False, na=False)]
print(flag_cats[['time', 'category', 'message_text']].to_string())

# 5. Telemetry: pick car #13 (winner) and peek
print(f"\n5. Telemetry sample (car #13):")
tele13 = pd.read_parquet(f'{R10}/telemetry/car_number=13/')
print(f"   shape: {tele13.shape}")
print(f"   dtypes:\n{tele13.dtypes.to_string()}")
print(f"   time range: {tele13.columns[tele13.dtypes.astype(str).str.contains('datetime')].tolist()}")
print(f"   head:\n{tele13.head(2).to_string()}")


=== ENERGY per lap ===  (energy/energy_per_lap.parquet)
shape: (852, 8)
dtypes:
car_number            int64
lap_number            int64
percent_consumed    float64
year                  int64
month                 int64
day                   int64
session              object
load_timestamp       object
head(3):
   car_number  lap_number  percent_consumed  year  month  day session                  load_timestamp
0           1           1               3.2  2024      5   12    Race  2024-11-03 11:23:16.631000 UTC
1           1           2               2.3  2024      5   12    Race  2024-11-03 11:23:16.631000 UTC
2           1           3               2.5  2024      5   12    Race  2024-11-03 11:23:16.631000 UTC

=== ATTACK events ===  (timing/attack.parquet)
shape: (129, 4)
dtypes:
active                          boolean
car_number                        Int64
eth_arrival_time    datetime64[ns, UTC]
scenario                          Int64
head(3):
   active  car_number                

In [ ]:
# Cell 4 — AM scenario investigation + race window anchors + lap join sanity check
import pandas as pd
import numpy as np

# --- Race anchors ---
green_flags = rc[rc['category']=='flag.green'].sort_values('time')
chequered   = rc[rc['category']=='flag.chequered'].sort_values('time')
RACE_START_UTC  = green_flags['time'].iloc[0]    # first green = lights out
RACE_END_UTC    = chequered['time'].iloc[0]
RACE_DURATION_S = (RACE_END_UTC - RACE_START_UTC).total_seconds()
SC_RESTART_UTC  = green_flags['time'].iloc[1] if len(green_flags) > 1 else None

print(f"Race start (green):    {RACE_START_UTC}")
print(f"Race end (chequered):  {RACE_END_UTC}")
print(f"Race duration:         {RACE_DURATION_S:.2f}s ({RACE_DURATION_S/60:.2f} min)")
print(f"SC restart green:      {SC_RESTART_UTC}")

# --- Per-car final lap & retirement detection ---
finals = energy.groupby('car_number').agg(
    final_lap=('lap_number','max'),
    energy_total=('percent_consumed','sum'),
).reset_index()
finals['finished'] = finals['energy_total'] >= 99.5
print(f"\n--- Per-car race summary ---")
print(finals.sort_values('final_lap', ascending=False).to_string(index=False))

# --- AM scenario investigation ---
print(f"\n--- AM scenario population pattern ---")
print(f"Total rows: {len(attack)}")
print(f"Rows with active=True:  {(attack['active']==True).sum()}")
print(f"Rows with active=False: {(attack['active']==False).sum()}")
print(f"Rows with active=NA:    {attack['active'].isna().sum()}")
print(f"Rows with scenario set: {attack['scenario'].notna().sum()}")
print(f"Scenario × active crosstab:")
print(pd.crosstab(attack['scenario'].fillna('NA'), attack['active'].fillna('NA'), dropna=False))

# Scenario presence by car
print(f"\nPer-car scenario values seen (across all that car's AM rows):")
for car, grp in attack.groupby('car_number'):
    scenarios = sorted(grp['scenario'].dropna().unique().tolist())
    activates = (grp['active']==True).sum()
    deactivates = (grp['active']==False).sum()
    print(f"  car {car:2d}: scenarios={scenarios}, activates={activates}, deactivates={deactivates}, total_rows={len(grp)}")

# --- Telemetry timezone fix + sanity ---
print(f"\n--- Telemetry tz handling (car 13) ---")
tele13_utc = tele13.copy()
tele13_utc['time'] = pd.to_datetime(tele13_utc['time']).dt.tz_localize('UTC')
print(f"Telemetry time range (UTC):  {tele13_utc['time'].min()} -> {tele13_utc['time'].max()}")
print(f"Sample rate check (first 100 rows):")
deltas = tele13_utc['time'].diff().dt.total_seconds().iloc[1:101]
print(f"  median delta: {deltas.median():.4f}s (expect 0.05 for 20Hz)")
print(f"  mean delta:   {deltas.mean():.4f}s")
print(f"  std:          {deltas.std():.4f}s")

# --- Lap join sanity check: car 13, lap 5 ---
print(f"\n--- Lap join sanity check: car 13, lap 5 ---")
c13_laps = laps[laps['car_number']==13].sort_values('lap_num')
lap5 = c13_laps[c13_laps['lap_num']==5].iloc[0]
lap5_start = lap5['start_time']
lap5_dur_s = lap5['time'] / 1000.0  # 'time' is lap duration in ms
lap5_end = lap5_start + pd.Timedelta(seconds=lap5_dur_s)
print(f"Lap 5: start={lap5_start}, duration={lap5_dur_s:.3f}s, end={lap5_end}")

window = tele13_utc[(tele13_utc['time'] >= lap5_start) & (tele13_utc['time'] < lap5_end)]
print(f"Telemetry samples in lap 5 window: {len(window)} (expect ~{int(lap5_dur_s*20)})")
print(f"Speed range in lap: {window['tv_speed'].min():.1f} - {window['tv_speed'].max():.1f} km/h")
print(f"Brake usage: pct samples with tv_brake>5: {(window['tv_brake']>5).mean()*100:.1f}%")

# --- Cars that emit AM-activate but no AM-deactivate (retirement during AM?) ---
print(f"\n--- AM activate/deactivate balance per car ---")
am_balance = attack.groupby('car_number')['active'].agg(
    activates=lambda x: (x==True).sum(),
    deactivates=lambda x: (x==False).sum(),
)
am_balance['delta'] = am_balance['activates'] - am_balance['deactivates']
print(am_balance.sort_values('delta', ascending=False).to_string())

Race start (green):    2024-05-12 13:04:05.726000+00:00
Race end (chequered):  2024-05-12 13:51:53.503000+00:00
Race duration:         2867.78s (47.80 min)
SC restart green:      2024-05-12 13:37:42.433000+00:00

--- Per-car race summary ---
 car_number  final_lap  energy_total  finished
          1         41         100.0      True
          2         41          99.2     False
          3         41         100.0      True
          4         41         100.0      True
          5         41         100.0      True
          8         41          99.8      True
         22         41         100.0      True
          9         41         100.0      True
         11         41         100.0      True
         13         41         100.0      True
         16         41          99.7      True
         17         41          97.4     False
         18         41         100.0      True
         21         41         100.0      True
         51         41          99.5      True
      

TypeError: Invalid value 'NA' for dtype Int64

In [ ]:
# Cell 4b — Fix the typing crash and finish the investigation
import pandas as pd

# Recompute finals with corrected "finished" logic (final_lap == max final_lap)
MAX_LAP = energy.groupby('car_number')['lap_number'].max().max()
finals = energy.groupby('car_number').agg(
    final_lap=('lap_number','max'),
    energy_total=('percent_consumed','sum'),
).reset_index()
finals['finished'] = finals['final_lap'] == MAX_LAP
print(f"Max lap reached by any car: {MAX_LAP}")
print(f"Finishers (final_lap=={MAX_LAP}): {finals['finished'].sum()}")
print(f"Retirees: {(~finals['finished']).sum()}")
print(finals.sort_values(['finished','final_lap'], ascending=[False,False]).to_string(index=False))

# --- AM scenario crosstab — cast to string so NA is handled ---
print(f"\n--- AM scenario × active (counts) ---")
print(pd.crosstab(
    attack['scenario'].astype('string').fillna('—'),
    attack['active'].astype('string').fillna('—'),
    dropna=False,
))

# Confirm scenario lives on its own row type
preview_rows = attack[attack['active'].isna()]
activate_rows = attack[attack['active']==True]
deactivate_rows = attack[attack['active']==False]
print(f"\nPreview rows (active=NA): {len(preview_rows)}, all have scenario? {preview_rows['scenario'].notna().all()}")
print(f"Activate rows: {len(activate_rows)}, scenario set on any? {activate_rows['scenario'].notna().any()}")
print(f"Deactivate rows: {len(deactivate_rows)}, scenario set on any? {deactivate_rows['scenario'].notna().any()}")

# Per-car scenario assignment (from preview rows)
print(f"\n--- Per-car scenario from preview rows ---")
car_scenarios = preview_rows.groupby('car_number')['scenario'].agg(['nunique','first','last','count']).reset_index()
car_scenarios.columns = ['car_number','distinct_scenarios','first_scenario','last_scenario','preview_count']
print(car_scenarios.to_string(index=False))
print(f"\nDistribution of distinct scenarios per car: {car_scenarios['distinct_scenarios'].value_counts().to_dict()}")
print(f"Total cars with any scenario preview: {len(car_scenarios)} / 22")

# Activation/deactivation balance — same pattern, typing-safe
print(f"\n--- AM activate/deactivate balance per car ---")
am_balance = attack.groupby('car_number').agg(
    activates=('active', lambda x: (x==True).sum()),
    deactivates=('active', lambda x: (x==False).sum()),
    previews=('active', lambda x: x.isna().sum()),
).reset_index()
am_balance['delta'] = am_balance['activates'] - am_balance['deactivates']
print(am_balance.sort_values('delta', ascending=False).to_string(index=False))

# --- Telemetry tz fix + 20Hz sanity ---
print(f"\n--- Telemetry tz handling (car 13) ---")
tele13_utc = tele13.copy()
tele13_utc['time'] = pd.to_datetime(tele13_utc['time']).dt.tz_localize('UTC')
print(f"Telemetry range: {tele13_utc['time'].min()} -> {tele13_utc['time'].max()}")
deltas = tele13_utc['time'].diff().dt.total_seconds().iloc[1:1001]
print(f"Sample rate (first 1000 deltas): median={deltas.median():.4f}s, mean={deltas.mean():.4f}s, std={deltas.std():.4f}s")
print(f"Gaps > 0.1s: {(deltas > 0.1).sum()}")

# --- Lap join sanity: car 13, lap 5 ---
print(f"\n--- Lap join sanity: car 13, lap 5 ---")
c13_laps = laps[laps['car_number']==13].sort_values('lap_num')
lap5 = c13_laps[c13_laps['lap_num']==5].iloc[0]
lap5_start = lap5['start_time']
lap5_dur_s = lap5['time'] / 1000.0
lap5_end = lap5_start + pd.Timedelta(seconds=lap5_dur_s)
window = tele13_utc[(tele13_utc['time'] >= lap5_start) & (tele13_utc['time'] < lap5_end)]
print(f"Lap 5: start={lap5_start}, duration={lap5_dur_s:.3f}s")
print(f"Telemetry in lap window: {len(window)} samples (expect ~{int(lap5_dur_s*20)})")
print(f"Speed range: {window['tv_speed'].min():.1f} - {window['tv_speed'].max():.1f} km/h")
print(f"Brake-on samples (>5%): {(window['tv_brake']>5).mean()*100:.1f}%")
print(f"Lap-5 energy: {energy[(energy['car_number']==13) & (energy['lap_number']==5)]['percent_consumed'].iloc[0]:.2f}%")

Max lap reached by any car: 41
Finishers (final_lap==41): 20
Retirees: 2
 car_number  final_lap  energy_total  finished
          1         41         100.0      True
          2         41          99.2      True
          3         41         100.0      True
          4         41         100.0      True
          5         41         100.0      True
          8         41          99.8      True
          9         41         100.0      True
         11         41         100.0      True
         13         41         100.0      True
         16         41          99.7      True
         17         41          97.4      True
         18         41         100.0      True
         21         41         100.0      True
         22         41         100.0      True
         25         41         100.0      True
         33         41         100.0      True
         37         41         100.0      True
         48         41         100.0      True
         51         41          99

In [ ]:
# Cell 5 — Lap windows, AM windows, per-car scenario state
import pandas as pd
import numpy as np

# Constants (locked from earlier cells)
RACE_START_UTC = pd.Timestamp('2024-05-12 13:04:05.726000+00:00')
RACE_END_UTC   = pd.Timestamp('2024-05-12 13:51:53.503000+00:00')
RACE_DURATION_S = (RACE_END_UTC - RACE_START_UTC).total_seconds()
DEFAULT_SCENARIO = 1   # build-doc default for cars w/ no preview rows

def race_time_s(utc_ts):
    """Convert any UTC timestamp to seconds since race start (negative pre-race)."""
    return (utc_ts - RACE_START_UTC).dt.total_seconds()

# ============================================================
# 1) LAP WINDOWS — per (car, lap) the start/end UTC + measured energy
# ============================================================
laps_sorted = laps.sort_values(['car_number','lap_num']).copy()
laps_sorted['lap_end_time'] = laps_sorted['start_time'] + pd.to_timedelta(laps_sorted['time'], unit='ms')
laps_sorted['lap_start_s'] = race_time_s(laps_sorted['start_time'])
laps_sorted['lap_end_s']   = race_time_s(laps_sorted['lap_end_time'])
laps_sorted['lap_duration_s'] = laps_sorted['time'] / 1000.0

# Join in per-lap measured energy
lap_windows = laps_sorted.merge(
    energy[['car_number','lap_number','percent_consumed']].rename(columns={'lap_number':'lap_num'}),
    on=['car_number','lap_num'],
    how='left',
)[['car_number','lap_num','lap_start_s','lap_end_s','lap_duration_s',
   'start_time','lap_end_time','percent_consumed','position','top_speed']]

print(f"=== LAP WINDOWS ===  shape={lap_windows.shape}")
print(f"Cars: {lap_windows['car_number'].nunique()}, laps per car: {lap_windows.groupby('car_number')['lap_num'].count().describe().to_dict()}")
print(f"Energy join — rows with NaN percent_consumed: {lap_windows['percent_consumed'].isna().sum()}")
print(f"Sample (car 13, all laps):")
print(lap_windows[lap_windows['car_number']==13][['lap_num','lap_start_s','lap_end_s','lap_duration_s','percent_consumed','position']].to_string(index=False))

# Validation: per-car energy sum from joined data
sum_check = lap_windows.groupby('car_number')['percent_consumed'].sum().sort_values(ascending=False)
print(f"\nPer-car energy sum from lap_windows (compare to earlier per-car):")
print(sum_check.to_string())

# ============================================================
# 2) AM WINDOWS — pair activate→deactivate events per car
# ============================================================
am_sorted = attack.sort_values(['car_number','eth_arrival_time']).copy()
am_sorted['t_s'] = race_time_s(am_sorted['eth_arrival_time'])

am_windows = []
for car, grp in am_sorted.groupby('car_number'):
    # Filter to activate/deactivate events only (drop preview rows)
    events = grp[grp['active'].notna()].copy().reset_index(drop=True)
    # Pair consecutive activate→deactivate
    pairs = []
    pending_start = None
    for _, row in events.iterrows():
        if row['active'] == True:
            pending_start = row
        elif row['active'] == False and pending_start is not None:
            pairs.append({
                'car_number': int(car),
                'activation_num': len(pairs) + 1,
                'start_utc': pending_start['eth_arrival_time'],
                'end_utc': row['eth_arrival_time'],
                'start_s': pending_start['t_s'],
                'end_s': row['t_s'],
                'duration_s': row['t_s'] - pending_start['t_s'],
            })
            pending_start = None
    am_windows.extend(pairs)

am_windows = pd.DataFrame(am_windows)
print(f"\n=== AM WINDOWS ===  shape={am_windows.shape}")
print(f"Cars with AM windows: {am_windows['car_number'].nunique()} (expect 22)")
print(f"Duration distribution:")
print(am_windows['duration_s'].describe())
print(f"\nSample (first 6 rows):")
print(am_windows.head(6).to_string(index=False))

# Sanity: duration should cluster near 60/120/180 (the scenario halves)
print(f"\nDuration histogram bins:")
print(pd.cut(am_windows['duration_s'], bins=[0,90,150,210,300]).value_counts().sort_index())

# ============================================================
# 3) CAR SCENARIO STATE — time-series of scenario selection
# ============================================================
preview_rows = attack[attack['active'].isna() & attack['scenario'].notna()].copy()
preview_rows['t_s'] = race_time_s(preview_rows['eth_arrival_time'])
car_scenario_timeline = preview_rows[['car_number','t_s','scenario','eth_arrival_time']].sort_values(['car_number','t_s'])

# Per-car final scenario (or default)
all_cars = sorted(drivers['car_number'].unique().tolist())
car_scenario_default = {}
for car in all_cars:
    car_previews = car_scenario_timeline[car_scenario_timeline['car_number']==car]
    if len(car_previews) > 0:
        car_scenario_default[car] = int(car_previews['scenario'].iloc[-1])
    else:
        car_scenario_default[car] = DEFAULT_SCENARIO

print(f"\n=== CAR SCENARIO STATE ===")
print(f"Scenario timeline rows: {len(car_scenario_timeline)}")
print(f"Cars with explicit scenario selection: {car_scenario_timeline['car_number'].nunique()} / 22")
print(f"\nFinal scenario per car (default={DEFAULT_SCENARIO} when no preview seen):")
for car in all_cars:
    explicit = car in car_scenario_timeline['car_number'].values
    marker = "" if explicit else " (default)"
    print(f"  car {car:3d}: scenario {car_scenario_default[car]}{marker}")

# Cross-check: AM duration sums per car should roughly match total AM budget for R10 (4 min = 240s)
am_totals = am_windows.groupby('car_number')['duration_s'].sum().sort_values()
print(f"\nTotal AM time used per car (expect ~240s = 4min R10 budget):")
print(am_totals.to_string())
print(f"Mean: {am_totals.mean():.1f}s, std: {am_totals.std():.1f}s")

=== LAP WINDOWS ===  shape=(854, 10)
Cars: 22, laps per car: {'count': 22.0, 'mean': 38.81818181818182, 'std': 7.384903285541155, 'min': 10.0, '25%': 41.0, '50%': 41.0, '75%': 41.0, 'max': 41.0}
Energy join — rows with NaN percent_consumed: 2
Sample (car 13, all laps):
 lap_num  lap_start_s  lap_end_s  lap_duration_s  percent_consumed  position
       1       -1.486     72.799          74.285               3.6         6
       2       72.799    140.647          67.848               2.3         6
       3      140.647    206.827           66.18               2.3         6
       4      206.827    273.145          66.318               2.3         6
       5      273.145    339.216          66.071               2.5         3
       6      339.216    405.012          65.796               2.5         2
       7      405.012    470.765          65.753               2.5         1
       8      470.765    538.074          67.309               2.4         3
       9      538.074    605.330     

In [ ]:
# Cell 6 — Energy model: identify NaN, define proxy, calibrate on car 13
import pandas as pd
import numpy as np

# --- A. Find the 2 NaN lap-energy rows ---
nan_rows = lap_windows[lap_windows['percent_consumed'].isna()][['car_number','lap_num','lap_start_s','lap_duration_s','position']]
print("=== NaN energy rows ===")
print(nan_rows.to_string(index=False))
print(f"\nThese are probably the retirement laps where the car was timed across but didn't complete the lap.")
print(f"For car 7 (retired lap 10): energy file ends at lap {energy[energy['car_number']==7]['lap_number'].max()}")
print(f"For car 23 (retired lap 24): energy file ends at lap {energy[energy['car_number']==23]['lap_number'].max()}")

# Decision: drop NaN rows from lap_windows (they have no energy anchor)
lap_windows_clean = lap_windows.dropna(subset=['percent_consumed']).copy()
print(f"\nlap_windows: {len(lap_windows)} → {len(lap_windows_clean)} after NaN drop")

# ============================================================
# B. Energy model — per-sample proxy
# ============================================================
M = 856.0           # kg with driver
G = 9.81            # m/s²
AM_BOOST = 350/300  # 1.167
REGEN_ETA = 0.70    # arbitrary, will be calibrated out anyway
DRAG_K = 0.4        # arbitrary
BRAKE_ON_PCT = 5    # threshold

def per_sample_proxy(v_ms, a_g, brake_pct, am_active):
    """Returns a signed instantaneous consumption proxy.
    Positive = drawing from battery. Negative = regen. Arbitrary units."""
    a = a_g * G  # convert g → m/s²
    # Tractive: only when accelerating and not on brakes
    if a > 0 and brake_pct < BRAKE_ON_PCT:
        p_tractive = M * v_ms * a
        if am_active:
            p_tractive *= AM_BOOST
    else:
        p_tractive = 0.0
    # Drag: always-on, scales with v²
    p_drag = DRAG_K * v_ms * v_ms
    # Regen: when braking and decelerating
    if brake_pct >= BRAKE_ON_PCT and a < 0:
        p_regen = REGEN_ETA * M * v_ms * a  # a is negative → p_regen negative
    else:
        p_regen = 0.0
    return p_tractive + p_drag + p_regen

# Vectorised version for speed
def per_sample_proxy_vec(v_kmh, a_g, brake_pct, am_active_bool):
    v_ms = v_kmh / 3.6
    a = a_g * G
    accelerating = (a > 0) & (brake_pct < BRAKE_ON_PCT)
    braking = (brake_pct >= BRAKE_ON_PCT) & (a < 0)
    p_tractive = np.where(accelerating, M * v_ms * a, 0.0)
    p_tractive = np.where(am_active_bool, p_tractive * AM_BOOST, p_tractive)
    p_drag = DRAG_K * v_ms * v_ms
    p_regen = np.where(braking, REGEN_ETA * M * v_ms * a, 0.0)
    return p_tractive + p_drag + p_regen

# ============================================================
# C. Pilot on car 13
# ============================================================
CAR = 13
tele_c = pd.read_parquet(f'{R10}/telemetry/car_number={CAR}/')
tele_c['time'] = pd.to_datetime(tele_c['time']).dt.tz_localize('UTC')
tele_c['t_s'] = (tele_c['time'] - RACE_START_UTC).dt.total_seconds()
tele_c = tele_c.sort_values('t_s').reset_index(drop=True)

# AM-active boolean per sample
am_c = am_windows[am_windows['car_number']==CAR][['start_s','end_s']]
tele_c['am_active'] = False
for _, w in am_c.iterrows():
    tele_c.loc[(tele_c['t_s'] >= w['start_s']) & (tele_c['t_s'] < w['end_s']), 'am_active'] = True
print(f"\nCar {CAR}: AM-active samples: {tele_c['am_active'].sum()} / {len(tele_c)} = {tele_c['am_active'].mean()*100:.1f}%")

# Raw proxy per sample
tele_c['raw_proxy'] = per_sample_proxy_vec(
    tele_c['tv_speed'].values,
    tele_c['tv_acc_x'].values,
    tele_c['tv_brake'].values,
    tele_c['am_active'].values,
)
# Per-sample dt (use actual delta-t to handle gaps)
tele_c['dt'] = tele_c['t_s'].diff().fillna(0.05).clip(upper=0.5)  # cap large gaps
tele_c['raw_e'] = tele_c['raw_proxy'] * tele_c['dt']

# Assign each sample to a lap
laps_c = lap_windows_clean[lap_windows_clean['car_number']==CAR].sort_values('lap_num')
def assign_lap(t_s, laps_df):
    matches = laps_df[(laps_df['lap_start_s'] <= t_s) & (t_s < laps_df['lap_end_s'])]
    return int(matches['lap_num'].iloc[0]) if len(matches) else -1
# Vectorised assignment via searchsorted
lap_starts = laps_c['lap_start_s'].values
lap_nums = laps_c['lap_num'].values
idx = np.searchsorted(lap_starts, tele_c['t_s'].values, side='right') - 1
in_range = (idx >= 0) & (idx < len(lap_nums))
tele_c['lap_num'] = -1
tele_c.loc[in_range, 'lap_num'] = lap_nums[idx[in_range]]

# Per-lap raw integral
lap_raw = tele_c[tele_c['lap_num']>0].groupby('lap_num')['raw_e'].sum().reset_index(name='raw_lap_integral')
lap_check = laps_c[['lap_num','lap_start_s','lap_end_s','lap_duration_s','percent_consumed']].merge(lap_raw, on='lap_num')
lap_check['scale'] = lap_check['percent_consumed'] / lap_check['raw_lap_integral']

print(f"\n=== Per-lap calibration scale, car {CAR} ===")
print(f"Scale stats: min={lap_check['scale'].min():.4e}, median={lap_check['scale'].median():.4e}, max={lap_check['scale'].max():.4e}")
print(f"Scale range (max/min): {lap_check['scale'].max()/lap_check['scale'].min():.2f}x")
print(f"Laps with negative raw_integral: {(lap_check['raw_lap_integral']<=0).sum()}")
print(f"Laps with wrong-sign scale (raw and pct disagree): {(lap_check['scale']<0).sum()}")

# Look at three interesting laps: normal, SC-affected, AM-active
print(f"\n=== Interesting laps, car {CAR} ===")
# Lap 5 = normal
# Lap 26 = biggest SC lap (110s duration)
# Find an AM-active lap
am_lap = None
for lap_num in laps_c['lap_num'].values:
    lap_t0 = laps_c[laps_c['lap_num']==lap_num]['lap_start_s'].iloc[0]
    lap_t1 = laps_c[laps_c['lap_num']==lap_num]['lap_end_s'].iloc[0]
    if any((am_c['start_s'] < lap_t1) & (am_c['end_s'] > lap_t0)):
        am_lap = lap_num
        break
print(f"First AM-active lap for car {CAR}: lap {am_lap}")
for L in [5, 26, am_lap]:
    r = lap_check[lap_check['lap_num']==L].iloc[0]
    print(f"  Lap {L:2d}: duration={r['lap_duration_s']:.2f}s, pct_consumed={r['percent_consumed']:.2f}, raw_integral={r['raw_lap_integral']:.3e}, scale={r['scale']:.3e}")

# Apply calibration: per-sample calibrated energy = raw_e * scale_for_that_lap
tele_c = tele_c.merge(lap_check[['lap_num','scale']], on='lap_num', how='left')
tele_c['cal_e'] = tele_c['raw_e'] * tele_c['scale'].fillna(0)

# Validate: cumulative at lap end should match cumulative anchor
tele_c['cum_pct_used'] = tele_c['cal_e'].cumsum()
print(f"\n=== Cumulative energy validation, car {CAR} ===")
expected_total = lap_check['percent_consumed'].sum()
got_total = tele_c['cum_pct_used'].iloc[-1]
print(f"Sum of percent_consumed (anchor): {expected_total:.3f}")
print(f"Sum of calibrated cumsum:         {got_total:.3f}")
print(f"Difference: {abs(expected_total - got_total):.6f} (should be ~0)")

# Check at every lap boundary
print(f"\nLap-by-lap cum check (first 10 and last 5):")
boundary_check = []
for _, lap_row in lap_check.iterrows():
    lap_end_idx = tele_c[tele_c['t_s'] < lap_row['lap_end_s']].index[-1]
    cum_at_end = tele_c.loc[lap_end_idx, 'cum_pct_used']
    expected_cum = lap_check[lap_check['lap_num']<=lap_row['lap_num']]['percent_consumed'].sum()
    boundary_check.append({
        'lap': lap_row['lap_num'],
        'cum_anchor': expected_cum,
        'cum_modeled': cum_at_end,
        'diff': cum_at_end - expected_cum,
    })
bc = pd.DataFrame(boundary_check)
print(pd.concat([bc.head(10), bc.tail(5)]).to_string(index=False))
print(f"\nMax abs diff at any lap boundary: {bc['diff'].abs().max():.4f}")

=== NaN energy rows ===
 car_number  lap_num  lap_start_s  lap_duration_s  position
         17        9      540.645            66.4         8
         17       10      607.045          66.959         8

These are probably the retirement laps where the car was timed across but didn't complete the lap.
For car 7 (retired lap 10): energy file ends at lap 10
For car 23 (retired lap 24): energy file ends at lap 24

lap_windows: 854 → 852 after NaN drop

Car 13: AM-active samples: 4470 / 61091 = 7.3%

=== Per-lap calibration scale, car 13 ===
Scale stats: min=8.7844e-07, median=1.3514e-06, max=2.4738e-06
Scale range (max/min): 2.82x
Laps with negative raw_integral: 0
Laps with wrong-sign scale (raw and pct disagree): 0

=== Interesting laps, car 13 ===
First AM-active lap for car 13: lap 8
  Lap  5: duration=66.07s, pct_consumed=2.50, raw_integral=2.021e+06, scale=1.237e-06
  Lap 26: duration=110.38s, pct_consumed=0.80, raw_integral=4.844e+05, scale=1.652e-06
  Lap  8: duration=67.31s, pct

In [ ]:
# Cell 6b — Fix lap assignment to bound by both start and end
import numpy as np
import pandas as pd

# Rebuild lap assignment with proper bounds
laps_c = lap_windows_clean[lap_windows_clean['car_number']==CAR].sort_values('lap_num').reset_index(drop=True)
lap_starts = laps_c['lap_start_s'].values
lap_ends   = laps_c['lap_end_s'].values
lap_nums   = laps_c['lap_num'].values

idx = np.searchsorted(lap_starts, tele_c['t_s'].values, side='right') - 1
idx_clipped = np.clip(idx, 0, len(lap_starts) - 1)
candidate_starts = lap_starts[idx_clipped]
candidate_ends   = lap_ends[idx_clipped]
t_arr = tele_c['t_s'].values
in_lap = (idx >= 0) & (t_arr >= candidate_starts) & (t_arr < candidate_ends)

# Clear previous assignment and re-assign
tele_c['lap_num'] = -1
tele_c.loc[in_lap, 'lap_num'] = lap_nums[idx_clipped[in_lap]]

print(f"Samples assigned to a lap: {(tele_c['lap_num']>0).sum()} / {len(tele_c)}")
print(f"Pre-race / inter-lap / post-race (lap_num=-1): {(tele_c['lap_num']==-1).sum()}")

# Recompute raw integrals, recalibrate, recheck
lap_raw = tele_c[tele_c['lap_num']>0].groupby('lap_num')['raw_e'].sum().reset_index(name='raw_lap_integral')
lap_check = laps_c[['lap_num','lap_start_s','lap_end_s','lap_duration_s','percent_consumed']].merge(lap_raw, on='lap_num')
lap_check['scale'] = lap_check['percent_consumed'] / lap_check['raw_lap_integral']

print(f"\n=== Recomputed calibration scales ===")
print(f"Scale stats: min={lap_check['scale'].min():.4e}, median={lap_check['scale'].median():.4e}, max={lap_check['scale'].max():.4e}")
print(f"Negative scales: {(lap_check['scale']<0).sum()}")
print(f"Range (max/min): {lap_check['scale'].max()/lap_check['scale'].min():.2f}x")

# Show the interesting laps again
print(f"\n=== Interesting laps, post-fix ===")
for L in [5, 26, 8, 41]:
    r = lap_check[lap_check['lap_num']==L].iloc[0]
    print(f"  Lap {L:2d}: duration={r['lap_duration_s']:.2f}s, pct_consumed={r['percent_consumed']:.2f}, raw_integral={r['raw_lap_integral']:.3e}, scale={r['scale']:.3e}")

# Recompute cal_e and cumsum
tele_c = tele_c.drop(columns=['scale','cal_e'], errors='ignore')
tele_c = tele_c.merge(lap_check[['lap_num','scale']], on='lap_num', how='left')
tele_c['cal_e'] = tele_c['raw_e'] * tele_c['scale'].fillna(0)
tele_c['cum_pct_used'] = tele_c['cal_e'].cumsum()

# Boundary check
print(f"\n=== Lap-boundary cumulative validation ===")
boundary_check = []
for _, lap_row in lap_check.iterrows():
    end_mask = tele_c['t_s'] < lap_row['lap_end_s']
    if end_mask.any():
        cum_at_end = tele_c.loc[end_mask, 'cum_pct_used'].iloc[-1]
        expected_cum = lap_check[lap_check['lap_num']<=lap_row['lap_num']]['percent_consumed'].sum()
        boundary_check.append({
            'lap': lap_row['lap_num'],
            'cum_anchor': expected_cum,
            'cum_modeled': cum_at_end,
            'diff': cum_at_end - expected_cum,
        })
bc = pd.DataFrame(boundary_check)
print(pd.concat([bc.head(5), bc.tail(5)]).to_string(index=False))
print(f"\nMax abs diff at any lap boundary: {bc['diff'].abs().max():.6f}")
print(f"Total samples included in calibration: {(tele_c['lap_num']>0).sum():,}")
print(f"Total samples excluded (pre/post/inter): {(tele_c['lap_num']==-1).sum():,}")

Samples assigned to a lap: 52455 / 61091
Pre-race / inter-lap / post-race (lap_num=-1): 8636

=== Recomputed calibration scales ===
Scale stats: min=8.7844e-07, median=1.3507e-06, max=1.8226e-06
Negative scales: 0
Range (max/min): 2.07x

=== Interesting laps, post-fix ===
  Lap  5: duration=66.07s, pct_consumed=2.50, raw_integral=2.021e+06, scale=1.237e-06
  Lap 26: duration=110.38s, pct_consumed=0.80, raw_integral=4.844e+05, scale=1.652e-06
  Lap  8: duration=67.31s, pct_consumed=2.40, raw_integral=2.732e+06, scale=8.784e-07
  Lap 41: duration=64.37s, pct_consumed=2.50, raw_integral=1.879e+06, scale=1.330e-06

=== Lap-boundary cumulative validation ===
 lap  cum_anchor  cum_modeled          diff
   1         3.6          3.6  8.881784e-16
   2         5.9          5.9 -5.329071e-15
   3         8.2          8.2 -1.598721e-14
   4        10.5         10.5  1.776357e-15
   5        13.0         13.0 -2.309264e-14
  37        88.7         88.7 -7.958079e-13
  38        91.6         91.6 

In [ ]:
# Cell 7 — Build calibrated 1Hz energy timeline for all 22 cars
import pandas as pd
import numpy as np
import time

def build_car_timeline(car_number, am_windows_df, lap_windows_df, race_start_utc, verbose=False):
    """Returns a 1Hz DataFrame for one car: race_time_s, energy_pct_used, am_active, in_lap, lap_num."""
    t0 = time.time()
    tele = pd.read_parquet(f'{R10}/telemetry/car_number={car_number}/')
    tele['time'] = pd.to_datetime(tele['time']).dt.tz_localize('UTC')
    tele['t_s'] = (tele['time'] - race_start_utc).dt.total_seconds()
    tele = tele.sort_values('t_s').reset_index(drop=True)

    # AM-active flag
    am_c = am_windows_df[am_windows_df['car_number']==car_number][['start_s','end_s']]
    tele['am_active'] = False
    for _, w in am_c.iterrows():
        tele.loc[(tele['t_s'] >= w['start_s']) & (tele['t_s'] < w['end_s']), 'am_active'] = True

    # Raw per-sample proxy
    tele['raw_proxy'] = per_sample_proxy_vec(
        tele['tv_speed'].values,
        tele['tv_acc_x'].values,
        tele['tv_brake'].values,
        tele['am_active'].values,
    )
    tele['dt'] = tele['t_s'].diff().fillna(0.05).clip(upper=0.5)
    tele['raw_e'] = tele['raw_proxy'] * tele['dt']

    # Lap assignment with proper bounds
    laps_c = lap_windows_df[lap_windows_df['car_number']==car_number].sort_values('lap_num').reset_index(drop=True)
    if len(laps_c) == 0:
        if verbose: print(f"  car {car_number}: no laps")
        return None
    lap_starts = laps_c['lap_start_s'].values
    lap_ends   = laps_c['lap_end_s'].values
    lap_nums   = laps_c['lap_num'].values
    idx = np.searchsorted(lap_starts, tele['t_s'].values, side='right') - 1
    idx_clipped = np.clip(idx, 0, len(lap_starts) - 1)
    cs = lap_starts[idx_clipped]; ce = lap_ends[idx_clipped]
    t_arr = tele['t_s'].values
    in_lap_mask = (idx >= 0) & (t_arr >= cs) & (t_arr < ce)
    tele['lap_num'] = -1
    tele.loc[in_lap_mask, 'lap_num'] = lap_nums[idx_clipped[in_lap_mask]]

    # Calibration scales
    lap_raw = tele[tele['lap_num']>0].groupby('lap_num')['raw_e'].sum().reset_index(name='raw_lap_integral')
    lap_check = laps_c[['lap_num','percent_consumed']].merge(lap_raw, on='lap_num', how='left')
    lap_check['scale'] = lap_check['percent_consumed'] / lap_check['raw_lap_integral']
    if (lap_check['scale']<0).any() or lap_check['scale'].isna().any():
        problem_laps = lap_check[(lap_check['scale']<0) | (lap_check['scale'].isna())]
        print(f"  car {car_number}: WARN bad scales for laps: {problem_laps['lap_num'].tolist()}")

    tele = tele.merge(lap_check[['lap_num','scale']], on='lap_num', how='left')
    tele['cal_e'] = tele['raw_e'] * tele['scale'].fillna(0)
    tele['cum_pct_used'] = tele['cal_e'].cumsum()

    # Boundary validation
    max_err = 0.0
    for _, lap_row in lap_check.iterrows():
        end_mask = tele['t_s'] < laps_c[laps_c['lap_num']==lap_row['lap_num']]['lap_end_s'].iloc[0]
        if end_mask.any():
            cum_at_end = tele.loc[end_mask, 'cum_pct_used'].iloc[-1]
            expected_cum = lap_check[lap_check['lap_num']<=lap_row['lap_num']]['percent_consumed'].sum()
            max_err = max(max_err, abs(cum_at_end - expected_cum))

    # Downsample to 1 Hz: take last sample of each integer second
    tele['t_int'] = tele['t_s'].astype(int)
    one_hz = tele.groupby('t_int').agg(
        race_time_s=('t_int', 'last'),
        energy_pct_used=('cum_pct_used', 'last'),
        am_active=('am_active', 'last'),
        lap_num=('lap_num', 'last'),
        speed_kmh=('tv_speed', 'last'),
        gps_lat=('tv_gps_lat', 'last'),
        gps_lng=('tv_gps_long', 'last'),
        gps_heading=('tv_gps_head', 'last'),
        accel_x=('tv_acc_x', 'last'),
        accel_y=('tv_acc_y', 'last'),
        brake_pct=('tv_brake', 'last'),
        steer=('tv_steer', 'last'),
        yaw_rate=('tv_yaw_rate', 'last'),
    ).reset_index(drop=True)
    one_hz['car_number'] = car_number
    one_hz['energy_pct_remaining'] = 100.0 - one_hz['energy_pct_used']

    if verbose:
        print(f"  car {car_number:3d}: {len(tele):>6,} raw → {len(one_hz):>5,} 1Hz | "
              f"max_boundary_err={max_err:.2e} | "
              f"laps={lap_check['lap_num'].max()} | t={time.time()-t0:.1f}s")
    return one_hz

# Build for all 22 cars
print("=== Building 1Hz timeline for all 22 cars ===\n")
all_cars = sorted(drivers['car_number'].unique().tolist())
timelines = {}
overall_t0 = time.time()
for car in all_cars:
    tl = build_car_timeline(car, am_windows, lap_windows_clean, RACE_START_UTC, verbose=True)
    if tl is not None:
        timelines[car] = tl

print(f"\nTotal build time: {time.time()-overall_t0:.1f}s")
print(f"Cars in timeline: {len(timelines)}")

# Stack and inspect
all_tl = pd.concat(timelines.values(), ignore_index=True)
print(f"\n=== Combined timeline ===")
print(f"Total rows: {len(all_tl):,}")
print(f"Race time range: {all_tl['race_time_s'].min()}s -> {all_tl['race_time_s'].max()}s")
print(f"Per-car row counts:")
print(all_tl.groupby('car_number').size().describe())

# Sanity: final energy_pct_used per car
print(f"\nFinal energy_pct_used per car (should match per-car energy_total):")
final_e = all_tl.sort_values('race_time_s').groupby('car_number').tail(1).sort_values('car_number')
print(final_e[['car_number','race_time_s','lap_num','energy_pct_used','energy_pct_remaining']].to_string(index=False))

=== Building 1Hz timeline for all 22 cars ===

  car   1: 59,141 raw → 3,318 1Hz | max_boundary_err=3.06e-13 | laps=41 | t=0.7s
  car   2: 57,163 raw → 3,318 1Hz | max_boundary_err=5.68e-13 | laps=41 | t=0.6s
  car   3: 59,064 raw → 3,318 1Hz | max_boundary_err=8.38e-13 | laps=41 | t=0.7s
  car 4: WARN bad scales for laps: [11]
  car   4: 60,185 raw → 3,317 1Hz | max_boundary_err=3.98e-13 | laps=41 | t=0.7s
  car 5: WARN bad scales for laps: [11, 26]
  car   5: 60,574 raw → 3,318 1Hz | max_boundary_err=3.13e-13 | laps=41 | t=0.8s
  car   7: 25,491 raw → 1,360 1Hz | max_boundary_err=3.91e-14 | laps=10 | t=0.6s
  car   8: 60,601 raw → 3,318 1Hz | max_boundary_err=5.68e-13 | laps=41 | t=0.8s
  car   9: 60,782 raw → 3,318 1Hz | max_boundary_err=9.38e-13 | laps=41 | t=0.7s
  car  11: 60,956 raw → 3,317 1Hz | max_boundary_err=4.97e-13 | laps=41 | t=0.7s
  car  13: 61,091 raw → 3,318 1Hz | max_boundary_err=8.10e-13 | laps=41 | t=0.6s
  car 16: WARN bad scales for laps: [26]
  car  16: 60,338 

In [ ]:
# Cell 7b — Diagnose the bad-scale laps
print("=== Bad-scale laps: car 4 lap 11, car 5 laps 11 & 26, car 16 lap 26, car 51 laps 11 & 26 ===\n")

bad_cases = [(4,11), (5,11), (5,26), (16,26), (51,11), (51,26)]

for car, lap in bad_cases:
    laps_c = lap_windows_clean[lap_windows_clean['car_number']==car]
    lap_row = laps_c[laps_c['lap_num']==lap].iloc[0]
    t0, t1 = lap_row['lap_start_s'], lap_row['lap_end_s']
    pct = lap_row['percent_consumed']

    # Reload telemetry for this car (small enough, no caching needed)
    tele = pd.read_parquet(f'{R10}/telemetry/car_number={car}/')
    tele['time'] = pd.to_datetime(tele['time']).dt.tz_localize('UTC')
    tele['t_s'] = (tele['time'] - RACE_START_UTC).dt.total_seconds()
    w = tele[(tele['t_s'] >= t0) & (tele['t_s'] < t1)]

    # Recompute raw proxy on this window
    am_c = am_windows[am_windows['car_number']==car][['start_s','end_s']]
    w_am = w['t_s'].apply(lambda t: any((am_c['start_s']<=t) & (am_c['end_s']>t)))
    raw = per_sample_proxy_vec(w['tv_speed'].values, w['tv_acc_x'].values,
                               w['tv_brake'].values, w_am.values)
    dt = w['t_s'].diff().fillna(0.05).clip(upper=0.5).values
    raw_e = raw * dt
    integral = raw_e.sum()

    print(f"Car {car} lap {lap}: duration={t1-t0:.1f}s, pct_consumed={pct}, samples={len(w)}")
    print(f"  speed range: {w['tv_speed'].min():.1f} - {w['tv_speed'].max():.1f} km/h")
    print(f"  accel_x range: {w['tv_acc_x'].min():.3f} - {w['tv_acc_x'].max():.3f} g")
    print(f"  brake range: {w['tv_brake'].min():.1f} - {w['tv_brake'].max():.1f}%")
    print(f"  raw integral: {integral:.4e}")
    print(f"  components: tractive={raw[raw>0].sum():.3e}, regen={raw[raw<0].sum():.3e}, drag-only_samples_count={((w['tv_acc_x']<=0) | (w['tv_brake']>=5)).sum()}")
    print()

=== Bad-scale laps: car 4 lap 11, car 5 laps 11 & 26, car 16 lap 26, car 51 laps 11 & 26 ===

Car 4 lap 11: duration=84.1s, pct_consumed=0.5, samples=1560
  speed range: 44.0 - 224.8 km/h
  accel_x range: -1.759 - 1.270 g
  brake range: 0.0 - 100.0%
  raw integral: -3.1834e+05
  components: tractive=5.847e+07, regen=-6.459e+07, drag-only_samples_count=1039

Car 5 lap 11: duration=84.4s, pct_consumed=0.7000000000000001, samples=1591
  speed range: 42.7 - 216.1 km/h
  accel_x range: -1.617 - 2.769 g
  brake range: 0.0 - 100.0%
  raw integral: -4.2230e+05
  components: tractive=5.598e+07, regen=-6.299e+07, drag-only_samples_count=1156

Car 5 lap 26: duration=109.1s, pct_consumed=0.5, samples=2081
  speed range: 28.7 - 181.1 km/h
  accel_x range: -1.106 - 2.055 g
  brake range: 0.0 - 100.0%
  raw integral: -1.4216e+05
  components: tractive=2.160e+07, regen=-2.239e+07, drag-only_samples_count=1489

Car 16 lap 26: duration=107.0s, pct_consumed=0.6000000000000001, samples=2035
  speed range:

In [ ]:
# Cell 7c — Patch the proxy with aux load + clamp negative lap integrals
import numpy as np
import pandas as pd
import time

AUX_FLOOR = 50.0   # always-on baseline, arbitrary units; calibration absorbs magnitude
MIN_LAP_INTEGRAL = 1e3  # floor for raw integral before computing scale (positive)

def per_sample_proxy_vec(v_kmh, a_g, brake_pct, am_active_bool):
    """Physics-informed per-sample energy draw proxy.
    Components: tractive, drag, regen, aux. Returns signed value (positive=draw, negative=regen)."""
    v_ms = v_kmh / 3.6
    a = a_g * G
    accelerating = (a > 0) & (brake_pct < BRAKE_ON_PCT)
    braking      = (brake_pct >= BRAKE_ON_PCT) & (a < 0)
    p_tractive = np.where(accelerating, M * v_ms * a, 0.0)
    p_tractive = np.where(am_active_bool, p_tractive * AM_BOOST, p_tractive)
    p_drag     = DRAG_K * v_ms * v_ms
    p_regen    = np.where(braking, REGEN_ETA * M * v_ms * a, 0.0)  # negative when braking
    return p_tractive + p_drag + p_regen + AUX_FLOOR


def build_car_timeline(car_number, am_windows_df, lap_windows_df, race_start_utc, verbose=False):
    t0 = time.time()
    tele = pd.read_parquet(f'{R10}/telemetry/car_number={car_number}/')
    tele['time'] = pd.to_datetime(tele['time']).dt.tz_localize('UTC')
    tele['t_s'] = (tele['time'] - race_start_utc).dt.total_seconds()
    tele = tele.sort_values('t_s').reset_index(drop=True)

    # AM-active flag — vectorized
    am_c = am_windows_df[am_windows_df['car_number']==car_number][['start_s','end_s']]
    am_active = np.zeros(len(tele), dtype=bool)
    t_arr = tele['t_s'].values
    for _, w in am_c.iterrows():
        am_active |= (t_arr >= w['start_s']) & (t_arr < w['end_s'])
    tele['am_active'] = am_active

    tele['raw_proxy'] = per_sample_proxy_vec(
        tele['tv_speed'].values, tele['tv_acc_x'].values,
        tele['tv_brake'].values, am_active,
    )
    tele['dt'] = tele['t_s'].diff().fillna(0.05).clip(upper=0.5)
    tele['raw_e'] = tele['raw_proxy'] * tele['dt']

    laps_c = lap_windows_df[lap_windows_df['car_number']==car_number].sort_values('lap_num').reset_index(drop=True)
    if len(laps_c) == 0:
        return None
    lap_starts = laps_c['lap_start_s'].values
    lap_ends   = laps_c['lap_end_s'].values
    lap_nums   = laps_c['lap_num'].values
    idx = np.searchsorted(lap_starts, t_arr, side='right') - 1
    idx_c = np.clip(idx, 0, len(lap_starts) - 1)
    in_lap = (idx >= 0) & (t_arr >= lap_starts[idx_c]) & (t_arr < lap_ends[idx_c])
    tele['lap_num'] = -1
    tele.loc[in_lap, 'lap_num'] = lap_nums[idx_c[in_lap]]

    # Per-lap raw integral with NEGATIVE CLAMP
    lap_raw = tele[tele['lap_num']>0].groupby('lap_num')['raw_e'].sum().reset_index(name='raw_integral_unclamped')
    lap_raw['raw_integral'] = lap_raw['raw_integral_unclamped'].clip(lower=MIN_LAP_INTEGRAL)
    lap_check = laps_c[['lap_num','percent_consumed']].merge(lap_raw, on='lap_num', how='left')
    lap_check['scale'] = lap_check['percent_consumed'] / lap_check['raw_integral']
    lap_check['was_clamped'] = lap_check['raw_integral_unclamped'] < MIN_LAP_INTEGRAL

    if (lap_check['scale']<0).any():
        bad = lap_check[lap_check['scale']<0]
        print(f"  car {car_number}: STILL bad scales for laps: {bad['lap_num'].tolist()}")

    n_clamped = lap_check['was_clamped'].sum()
    if n_clamped > 0 and verbose:
        clamped_laps = lap_check[lap_check['was_clamped']]['lap_num'].tolist()
        print(f"  car {car_number}: clamped {n_clamped} lap(s): {clamped_laps}")

    tele = tele.merge(lap_check[['lap_num','scale']], on='lap_num', how='left')

    # If a lap was clamped, redistribute proportionally to time-in-lap (not raw_e)
    # because raw_e is net-negative there. Use a flat consumption rate for clamped laps.
    clamped_lap_nums = set(lap_check[lap_check['was_clamped']]['lap_num'].tolist())
    if clamped_lap_nums:
        clamped_mask = tele['lap_num'].isin(clamped_lap_nums)
        # For clamped laps: replace cal_e with even distribution
        # = pct_consumed / lap_duration * dt
        lap_dur_lookup = laps_c.set_index('lap_num')['lap_duration_s'].to_dict()
        lap_pct_lookup = lap_check.set_index('lap_num')['percent_consumed'].to_dict()
        even_rate = tele['lap_num'].map(lambda L: lap_pct_lookup.get(L,0)/lap_dur_lookup.get(L,1) if L in clamped_lap_nums else 0)
        tele['cal_e'] = np.where(clamped_mask, even_rate * tele['dt'], tele['raw_e'] * tele['scale'].fillna(0))
    else:
        tele['cal_e'] = tele['raw_e'] * tele['scale'].fillna(0)

    tele['cum_pct_used'] = tele['cal_e'].cumsum()

    # Boundary validation
    max_err = 0.0
    for _, lap_row in lap_check.iterrows():
        end_t = laps_c[laps_c['lap_num']==lap_row['lap_num']]['lap_end_s'].iloc[0]
        end_mask = tele['t_s'] < end_t
        if end_mask.any():
            cum_at_end = tele.loc[end_mask, 'cum_pct_used'].iloc[-1]
            expected = lap_check[lap_check['lap_num']<=lap_row['lap_num']]['percent_consumed'].sum()
            max_err = max(max_err, abs(cum_at_end - expected))

    # Downsample to 1 Hz
    tele['t_int'] = tele['t_s'].astype(int)
    one_hz = tele.groupby('t_int').agg(
        race_time_s=('t_int','last'),
        energy_pct_used=('cum_pct_used','last'),
        am_active=('am_active','last'),
        lap_num=('lap_num','last'),
        speed_kmh=('tv_speed','last'),
        gps_lat=('tv_gps_lat','last'),
        gps_lng=('tv_gps_long','last'),
        gps_heading=('tv_gps_head','last'),
        accel_x=('tv_acc_x','last'),
        accel_y=('tv_acc_y','last'),
        brake_pct=('tv_brake','last'),
        steer=('tv_steer','last'),
        yaw_rate=('tv_yaw_rate','last'),
    ).reset_index(drop=True)
    one_hz['car_number'] = car_number
    one_hz['energy_pct_remaining'] = 100.0 - one_hz['energy_pct_used']

    if verbose:
        print(f"  car {car_number:3d}: {len(tele):>6,} raw → {len(one_hz):>5,} 1Hz | "
              f"max_err={max_err:.2e} | clamped={n_clamped} | t={time.time()-t0:.1f}s")
    return one_hz

print("=== Rebuilding all 22 cars with patched proxy ===\n")
overall_t0 = time.time()
timelines = {}
for car in sorted(drivers['car_number'].unique().tolist()):
    tl = build_car_timeline(car, am_windows, lap_windows_clean, RACE_START_UTC, verbose=True)
    if tl is not None:
        timelines[car] = tl
print(f"\nTotal: {time.time()-overall_t0:.1f}s")

# Reaggregate
all_tl = pd.concat(timelines.values(), ignore_index=True)
print(f"\n=== Combined timeline ===  rows={len(all_tl):,}")
final_e = all_tl.sort_values('race_time_s').groupby('car_number').tail(1).sort_values('car_number')
print(final_e[['car_number','race_time_s','energy_pct_used','energy_pct_remaining']].to_string(index=False))

=== Rebuilding all 22 cars with patched proxy ===

  car   1: 59,141 raw → 3,318 1Hz | max_err=8.24e-13 | clamped=0 | t=0.7s
  car   2: 57,163 raw → 3,318 1Hz | max_err=8.95e-13 | clamped=0 | t=0.6s
  car   3: 59,064 raw → 3,318 1Hz | max_err=9.09e-13 | clamped=0 | t=0.8s
  car 4: clamped 1 lap(s): [11]
  car   4: 60,185 raw → 3,317 1Hz | max_err=1.19e-04 | clamped=1 | t=0.6s
  car 5: clamped 2 lap(s): [11, 26]
  car   5: 60,574 raw → 3,318 1Hz | max_err=2.79e-04 | clamped=2 | t=0.7s
  car   7: 25,491 raw → 1,360 1Hz | max_err=3.55e-14 | clamped=0 | t=0.6s
  car   8: 60,601 raw → 3,318 1Hz | max_err=5.12e-13 | clamped=0 | t=0.7s
  car   9: 60,782 raw → 3,318 1Hz | max_err=1.19e-12 | clamped=0 | t=0.6s
  car  11: 60,956 raw → 3,317 1Hz | max_err=8.10e-13 | clamped=0 | t=0.7s
  car  13: 61,091 raw → 3,318 1Hz | max_err=3.41e-13 | clamped=0 | t=0.6s
  car 16: clamped 1 lap(s): [26]
  car  16: 60,338 raw → 3,318 1Hz | max_err=3.93e-05 | clamped=1 | t=0.7s
  car  17: 59,880 raw → 3,317 1Hz 

In [ ]:
# Cell 8 — Position over time, scenario state, race phase
import pandas as pd
import numpy as np

# ============================================================
# 1) POSITION OVER TIME
# ============================================================
# Two sources of position change:
#   a) Lap completion: position at end of each lap (from laps.parquet)
#   b) Mid-lap overtakes: timing.parquet rows where position_change is set
# Strategy: build a per-car position timeline with both, then forward-fill at 1Hz.

# (a) lap-end positions
lap_pos = lap_windows[['car_number','lap_end_s','position']].copy()
lap_pos = lap_pos.rename(columns={'lap_end_s':'t_s','position':'pos'})
lap_pos['source'] = 'lap_end'
lap_pos = lap_pos.dropna(subset=['pos'])

# (b) overtake events from timing.parquet
# An overtake gives us the FOR car's position change. We don't have absolute position from these
# directly, but we know position_change. We'll use them as event markers and derive position by
# accumulating changes since the last lap-boundary anchor.
overtake_events = timing[timing['position_change'].notna()].copy()
overtake_events['t_s'] = (overtake_events['eth_arrival_time'] - RACE_START_UTC).dt.total_seconds()
print(f"Overtake events in R10: {len(overtake_events)}")
print(f"Sample:\n{overtake_events[['t_s','car_number','participant','position_change']].head(8).to_string()}")

# Build a per-car position timeline:
# Walk forward from each lap_end anchor, applying position_change events until the next anchor.
def build_position_timeline(car):
    car_lap_pos = lap_pos[lap_pos['car_number']==car].sort_values('t_s')
    car_overtakes = overtake_events[overtake_events['car_number']==car].sort_values('t_s')
    rows = []
    # Start with lap-end anchors as known-truth points
    for _, lap_row in car_lap_pos.iterrows():
        rows.append({'t_s': lap_row['t_s'], 'position': int(lap_row['pos']), 'source':'lap_end'})
    # Add overtake events that shifted this car; we'll resolve position by interpolation
    for _, ov in car_overtakes.iterrows():
        rows.append({'t_s': ov['t_s'], 'position_change': int(ov['position_change']), 'source':'overtake'})
    df = pd.DataFrame(rows).sort_values('t_s').reset_index(drop=True)
    df['car_number'] = car

    # Resolve overtake positions by walking forward from last known position
    last_pos = None
    positions = []
    for _, r in df.iterrows():
        if r['source'] == 'lap_end':
            last_pos = int(r['position'])
            positions.append(last_pos)
        else:
            if last_pos is not None:
                last_pos = last_pos - int(r['position_change'])  # gaining position = lower number
                positions.append(last_pos)
            else:
                positions.append(None)
    df['resolved_position'] = positions
    return df[['car_number','t_s','resolved_position','source']].dropna()

print("\nBuilding position timelines for all cars...")
position_timelines = pd.concat(
    [build_position_timeline(c) for c in sorted(drivers['car_number'].unique().tolist())],
    ignore_index=True,
)
print(f"Total position events: {len(position_timelines)}")
print(f"Sample (car 13):\n{position_timelines[position_timelines['car_number']==13].head(10).to_string(index=False)}")

# ============================================================
# 2) SCENARIO STATE OVER TIME
# ============================================================
# preview rows give scenario at a point in time; forward-fill per car; default=1 for cars w/o previews
preview_events = attack[attack['active'].isna() & attack['scenario'].notna()].copy()
preview_events['t_s'] = (preview_events['eth_arrival_time'] - RACE_START_UTC).dt.total_seconds()
preview_events = preview_events[['car_number','t_s','scenario']].sort_values(['car_number','t_s'])
print(f"\nScenario change events: {len(preview_events)}")
print(f"Cars with scenario events: {preview_events['car_number'].nunique()}")
print(f"Sample (car 13 scenario history):\n{preview_events[preview_events['car_number']==13].to_string(index=False)}")

# ============================================================
# 3) RACE PHASE TIMELINE (single global series, not per-car)
# ============================================================
# Map RC categories to phase transitions
rc_sorted = rc.sort_values('time').copy()
rc_sorted['t_s'] = (rc_sorted['time'] - RACE_START_UTC).dt.total_seconds()

def category_to_phase(cat, msg):
    if cat == 'flag.green':
        return 'racing'
    if cat == 'flag.chequered':
        return 'chequered'
    if cat == 'sc.deployed':
        return 'safety_car'
    if cat == 'sc.in_this_lap':
        return None  # transition, not a phase
    if cat == 'flag.double_yellow_at_turn':
        return 'full_course_yellow' if 'TURN 3' in (msg or '') else None  # rough heuristic
    return None  # don't change phase

phase_transitions = []
current_phase = 'pre_race'
phase_transitions.append({'t_s': -10000, 'phase': current_phase, 'trigger': 'init'})
for _, r in rc_sorted.iterrows():
    new_phase = category_to_phase(r['category'], r['message_text'])
    if new_phase and new_phase != current_phase:
        phase_transitions.append({'t_s': r['t_s'], 'phase': new_phase, 'trigger': r['category']})
        current_phase = new_phase
phase_timeline = pd.DataFrame(phase_transitions)
print(f"\n=== Race phase transitions ===")
print(phase_timeline.to_string(index=False))

# ============================================================
# 4) Forward-fill all three onto the 1Hz grid in all_tl
# ============================================================
# Sort once
all_tl_sorted = all_tl.sort_values(['car_number','race_time_s']).reset_index(drop=True)

# Position: per-car, merge_asof with backward direction
pos_for_merge = position_timelines.sort_values(['car_number','t_s'])[['car_number','t_s','resolved_position']]
pos_for_merge['t_s'] = pos_for_merge['t_s'].astype(float)
all_tl_sorted['race_time_s'] = all_tl_sorted['race_time_s'].astype(float)
all_tl_sorted = pd.merge_asof(
    all_tl_sorted.sort_values('race_time_s'),
    pos_for_merge.sort_values('t_s').rename(columns={'t_s':'race_time_s'}),
    on='race_time_s', by='car_number', direction='backward',
)

# Scenario: per-car, merge_asof
sce_for_merge = preview_events[['car_number','t_s','scenario']].sort_values(['car_number','t_s'])
sce_for_merge['t_s'] = sce_for_merge['t_s'].astype(float)
all_tl_sorted = pd.merge_asof(
    all_tl_sorted.sort_values('race_time_s'),
    sce_for_merge.sort_values('t_s').rename(columns={'t_s':'race_time_s'}),
    on='race_time_s', by='car_number', direction='backward',
)
all_tl_sorted['scenario'] = all_tl_sorted['scenario'].fillna(1).astype(int)  # default=1 per build doc

# Race phase: global, merge_asof without by
phase_for_merge = phase_timeline[['t_s','phase']].sort_values('t_s').rename(columns={'t_s':'race_time_s'})
phase_for_merge['race_time_s'] = phase_for_merge['race_time_s'].astype(float)
all_tl_sorted = pd.merge_asof(
    all_tl_sorted.sort_values('race_time_s'),
    phase_for_merge, on='race_time_s', direction='backward',
)

# Sort back to natural order
all_tl_sorted = all_tl_sorted.sort_values(['car_number','race_time_s']).reset_index(drop=True)

print(f"\n=== Enriched timeline ===")
print(f"Total rows: {len(all_tl_sorted):,}")
print(f"Null position rows: {all_tl_sorted['resolved_position'].isna().sum()}")
print(f"Null scenario rows: {all_tl_sorted['scenario'].isna().sum()}")
print(f"Null phase rows:    {all_tl_sorted['phase'].isna().sum()}")
print(f"Phase distribution:\n{all_tl_sorted['phase'].value_counts().to_string()}")
print(f"\nSample car 13 around race start (race_time 0-30):")
print(all_tl_sorted[(all_tl_sorted['car_number']==13) &
                   (all_tl_sorted['race_time_s'].between(0,30))]
      [['race_time_s','lap_num','resolved_position','scenario','phase','energy_pct_remaining','am_active']].head(15).to_string(index=False))

Overtake events in R10: 1290
Sample:
      t_s  car_number participant  position_change
0   6.065           1          37               -1
1   6.065           2           1                1
5   6.519           5          94               -1
6   6.519           6           7                1
7   7.003           6          48               -1
8   7.003           7           7                1
9   7.124           7           2               -1
10  7.124           8           7                1

Building position timelines for all cars...
Total position events: 1658
Sample (car 13):
 car_number     t_s  resolved_position   source
         13  72.799                6.0  lap_end
         13 140.647                6.0  lap_end
         13 155.331                5.0 overtake
         13 166.306                5.0 overtake
         13 206.827                6.0  lap_end
         13 222.614                5.0 overtake
         13 232.750                5.0 overtake
         13 273.145           

MergeError: incompatible merge keys [0] dtype('int64') and Int64Dtype(), must be the same type

In [ ]:
# Cell 8b — Normalize dtypes and finish the enrichment merges
import pandas as pd
import numpy as np

# Normalize car_number to plain int64 in all four frames before merging
all_tl_sorted = all_tl.copy()
all_tl_sorted['car_number'] = all_tl_sorted['car_number'].astype('int64')
all_tl_sorted['race_time_s'] = all_tl_sorted['race_time_s'].astype('float64')
all_tl_sorted = all_tl_sorted.sort_values(['car_number','race_time_s']).reset_index(drop=True)

pos_for_merge = position_timelines[['car_number','t_s','resolved_position']].copy()
pos_for_merge['car_number'] = pos_for_merge['car_number'].astype('int64')
pos_for_merge['t_s'] = pos_for_merge['t_s'].astype('float64')
pos_for_merge = pos_for_merge.rename(columns={'t_s':'race_time_s'}).sort_values(['car_number','race_time_s'])

sce_for_merge = preview_events[['car_number','t_s','scenario']].copy()
sce_for_merge['car_number'] = sce_for_merge['car_number'].astype('int64')
sce_for_merge['t_s'] = sce_for_merge['t_s'].astype('float64')
sce_for_merge['scenario'] = sce_for_merge['scenario'].astype('int64')
sce_for_merge = sce_for_merge.rename(columns={'t_s':'race_time_s'}).sort_values(['car_number','race_time_s'])

phase_for_merge = phase_timeline[['t_s','phase']].copy()
phase_for_merge['t_s'] = phase_for_merge['t_s'].astype('float64')
phase_for_merge = phase_for_merge.rename(columns={'t_s':'race_time_s'}).sort_values('race_time_s')

# merge_asof requires the left frame sorted by the `on` key (race_time_s), not by `by` (car_number).
# Do all three merges with the LEFT sorted by race_time_s only.
def asof_merge_by_car(left, right, on='race_time_s', by='car_number'):
    return pd.merge_asof(
        left.sort_values(on),
        right.sort_values(on),
        on=on, by=by, direction='backward'
    )

# Position merge
enriched = asof_merge_by_car(all_tl_sorted, pos_for_merge)
# Scenario merge
enriched = asof_merge_by_car(enriched, sce_for_merge)
enriched['scenario'] = enriched['scenario'].fillna(1).astype(int)
# Race phase merge (global, no by=)
enriched = pd.merge_asof(
    enriched.sort_values('race_time_s'),
    phase_for_merge.sort_values('race_time_s'),
    on='race_time_s', direction='backward'
)

# Restore natural sort
enriched = enriched.sort_values(['car_number','race_time_s']).reset_index(drop=True)

print(f"=== Enriched timeline ===  rows={len(enriched):,}")
print(f"Null position rows: {enriched['resolved_position'].isna().sum()}")
print(f"Null scenario rows: {enriched['scenario'].isna().sum()}")
print(f"Null phase rows:    {enriched['phase'].isna().sum()}")
print(f"\nPhase distribution:")
print(enriched['phase'].value_counts().to_string())

# Position null distribution — should be only pre-race
null_pos = enriched[enriched['resolved_position'].isna()]
print(f"\nRows with null position: {len(null_pos)} (mostly pre-race?)")
print(f"  race_time range: {null_pos['race_time_s'].min()}s - {null_pos['race_time_s'].max()}s")
print(f"  cars affected: {sorted(null_pos['car_number'].unique().tolist())}")

# Spot-check car 13 around green flag and around SC
print(f"\n=== Car 13: first 15 race-seconds ===")
print(enriched[(enriched['car_number']==13) & (enriched['race_time_s'].between(0,15))]
      [['race_time_s','lap_num','resolved_position','scenario','phase','energy_pct_remaining','am_active','speed_kmh']].to_string(index=False))

print(f"\n=== Car 13: around SC deployment at t=726 ===")
print(enriched[(enriched['car_number']==13) & (enriched['race_time_s'].between(720,740))]
      [['race_time_s','lap_num','resolved_position','phase','energy_pct_remaining','speed_kmh']].to_string(index=False))

print(f"\n=== Car 13: lap 12 (the 102.8s SC lap) ===")
print(enriched[(enriched['car_number']==13) & (enriched['lap_num']==12)]
      [['race_time_s','resolved_position','phase','energy_pct_remaining','speed_kmh','brake_pct']].head(10).to_string(index=False))
print("...")
print(enriched[(enriched['car_number']==13) & (enriched['lap_num']==12)]
      [['race_time_s','resolved_position','phase','energy_pct_remaining','speed_kmh','brake_pct']].tail(5).to_string(index=False))

# Validate against R10 finishing positions
print(f"\n=== Final positions per car (last lap completion) ===")
final_pos = enriched.dropna(subset=['resolved_position']).sort_values('race_time_s').groupby('car_number').tail(1)
final_pos = final_pos.sort_values('resolved_position')
print(final_pos[['car_number','resolved_position','race_time_s','lap_num','energy_pct_remaining']].to_string(index=False))

=== Enriched timeline ===  rows=70,770
Null position rows: 9718
Null scenario rows: 0
Null phase rows:    0

Phase distribution:
phase
racing                33685
safety_car            20596
pre_race               8030
full_course_yellow     6762
chequered              1697

Rows with null position: 9718 (mostly pre-race?)
  race_time range: -365.0s - 105.0s
  cars affected: [1, 2, 3, 4, 5, 7, 8, 9, 11, 13, 16, 17, 18, 21, 22, 23, 25, 33, 37, 48, 51, 94]

=== Car 13: first 15 race-seconds ===
 race_time_s  lap_num  resolved_position  scenario  phase  energy_pct_remaining  am_active  speed_kmh
         0.0        1                NaN         2 racing             99.816237      False      69.79
         1.0        1                NaN         2 racing             99.595643      False     100.47
         2.0        1                NaN         2 racing             99.316385      False     130.05
         3.0        1                NaN         2 racing             99.071647      False    

In [ ]:
# Cell 8c — Simpler, drift-free position resolution from lap-end anchors only
import pandas as pd
import numpy as np

# Just use lap_pos (lap-end positions) — drop the overtake reconstruction entirely
lap_pos_clean = lap_windows[['car_number','lap_end_s','position']].copy()
lap_pos_clean = lap_pos_clean.rename(columns={'lap_end_s':'race_time_s','position':'resolved_position'})
lap_pos_clean = lap_pos_clean.dropna(subset=['resolved_position'])
lap_pos_clean['car_number'] = lap_pos_clean['car_number'].astype('int64')
lap_pos_clean['race_time_s'] = lap_pos_clean['race_time_s'].astype('float64')
lap_pos_clean['resolved_position'] = lap_pos_clean['resolved_position'].astype('int64')

# Drop the old position column from enriched, re-merge with the clean version
enriched_v2 = enriched.drop(columns=['resolved_position']).copy()
enriched_v2 = pd.merge_asof(
    enriched_v2.sort_values('race_time_s'),
    lap_pos_clean.sort_values('race_time_s'),
    on='race_time_s', by='car_number', direction='backward'
)
enriched_v2 = enriched_v2.sort_values(['car_number','race_time_s']).reset_index(drop=True)

# Pre-race / grid positions: use the starting grid for the pre-race window
from pandas import read_parquet
startgrid = pd.read_parquet(f'{R10}/timing/startgrid.parquet')
startgrid['car_number'] = startgrid['car_number'].astype('int64')
grid_pos = startgrid.set_index('car_number')['position'].to_dict()
print(f"Starting grid:\n{startgrid.sort_values('position').to_string(index=False)}")

# Fill null positions (pre-race) with starting grid
pre_race_mask = enriched_v2['resolved_position'].isna()
enriched_v2.loc[pre_race_mask, 'resolved_position'] = (
    enriched_v2.loc[pre_race_mask, 'car_number'].map(grid_pos)
)
enriched_v2['resolved_position'] = enriched_v2['resolved_position'].astype('int64')

# Validation
print(f"\n=== Enriched v2 ===  rows={len(enriched_v2):,}")
print(f"Null positions remaining: {enriched_v2['resolved_position'].isna().sum()}")
print(f"Position range: {enriched_v2['resolved_position'].min()} - {enriched_v2['resolved_position'].max()}")
print(f"Position value counts:\n{enriched_v2['resolved_position'].value_counts().sort_index().to_string()}")

# Final positions
final_v2 = enriched_v2.sort_values('race_time_s').groupby('car_number').tail(1).sort_values('resolved_position')
print(f"\n=== Final positions per car (v2) ===")
print(final_v2[['car_number','resolved_position','race_time_s','lap_num']].to_string(index=False))

# Sanity: at any single race-second, the position values across all cars should be 1..22 (or fewer if retirees)
print(f"\n=== Position uniqueness check at race_time=1500 ===")
snap = enriched_v2[enriched_v2['race_time_s']==1500].sort_values('resolved_position')
print(snap[['car_number','resolved_position','lap_num','energy_pct_remaining']].to_string(index=False))
print(f"\nDistinct positions in snapshot: {snap['resolved_position'].nunique()} / {len(snap)} cars")
print(f"Position range in snapshot: {snap['resolved_position'].min()} - {snap['resolved_position'].max()}")

# Car 13 around SC, redone
print(f"\n=== Car 13 around SC deployment (v2) ===")
print(enriched_v2[(enriched_v2['car_number']==13) & (enriched_v2['race_time_s'].between(720,740))]
      [['race_time_s','lap_num','resolved_position','phase','energy_pct_remaining','speed_kmh']].to_string(index=False))

Starting grid:
 position  car_number  penalized
        1           1      False
        2          37      False
        3          17      False
        4           9      False
        5           7      False
        6          94      False
        7          48      False
        8           2       True
        9          25      False
       10          13      False
       11           4      False
       12          11      False
       13          18      False
       14           5      False
       15           3      False
       16          22      False
       17          23      False
       18           8      False
       19          16      False
       20          21      False
       21          51      False
       22          33       True

=== Enriched v2 ===  rows=70,770
Null positions remaining: 0
Position range: 1 - 22
Position value counts:
resolved_position
1     3331
2     3379
3     3251
4     3315
5     3387
6     3318
7     4323
8     3344
9     3556
1

In [ ]:
# Cell 9 — Build per-tick frames with events, retirement flags, AM budget, then write JSONL.gz
import pandas as pd
import numpy as np
import json
import gzip
import io

# ============================================================
# Build per-car retirement, driver name, and AM budget lookup
# ============================================================
final_laps = energy.groupby('car_number')['lap_number'].max().to_dict()
driver_names = drivers.set_index('car_number')['driver_short_name'].to_dict()
print(f"Retirement final laps: {final_laps}")

# Lap-end times per car for retirement detection
last_lap_end_time = lap_windows.sort_values(['car_number','lap_num']).groupby('car_number')['lap_end_s'].max().to_dict()

# ============================================================
# Prepare event sources, all keyed to race_time_s
# ============================================================
# AM activation events
am_activate_events = am_windows[['car_number','start_s','end_s','duration_s','activation_num']].copy()
am_activate_events = am_activate_events.rename(columns={'start_s':'t_s'})
am_activate_events['type'] = 'attack_mode_activated'
am_activate_events['t_int'] = am_activate_events['t_s'].astype(int)

am_deactivate_events = am_windows[['car_number','end_s','activation_num']].copy().rename(columns={'end_s':'t_s'})
am_deactivate_events['type'] = 'attack_mode_deactivated'
am_deactivate_events['t_int'] = am_deactivate_events['t_s'].astype(int)

# Overtake events (filter to ones inside race window)
ot_events = overtake_events[['car_number','participant','position_change','t_s']].copy()
ot_events = ot_events[(ot_events['t_s']>=0) & (ot_events['t_s']<=RACE_DURATION_S)]
ot_events['type'] = 'overtake'
ot_events['t_int'] = ot_events['t_s'].astype(int)

# Race control events
rc_events = rc_sorted[['t_s','category','message_text','attrs_json']].copy()
rc_events = rc_events[(rc_events['t_s']>=-365) & (rc_events['t_s']<=RACE_DURATION_S+30)]
rc_events['type'] = 'race_control'
rc_events['t_int'] = rc_events['t_s'].astype(int)

# Lap-completion events (use lap_end_s from lap_windows_clean)
lap_complete_events = lap_windows_clean[['car_number','lap_num','lap_end_s','lap_duration_s','top_speed']].copy()
lap_complete_events = lap_complete_events.rename(columns={'lap_end_s':'t_s'})
lap_complete_events['type'] = 'lap_completed'
lap_complete_events['t_int'] = lap_complete_events['t_s'].astype(int)

print(f"\nEvent counts: am_activate={len(am_activate_events)}, am_deactivate={len(am_deactivate_events)}, "
      f"overtake={len(ot_events)}, race_control={len(rc_events)}, lap_complete={len(lap_complete_events)}")

# ============================================================
# Build per-tick events bucket
# ============================================================
def to_event_dicts(df, kind):
    out = {}
    for _, r in df.iterrows():
        t_int = int(r['t_int'])
        if kind == 'attack_mode_activated':
            e = {'type':'attack_mode_activated','car_number':int(r['car_number']),
                 'duration_s': round(float(r['duration_s']),1), 'activation_num': int(r['activation_num'])}
        elif kind == 'attack_mode_deactivated':
            e = {'type':'attack_mode_deactivated','car_number':int(r['car_number']),
                 'activation_num': int(r['activation_num'])}
        elif kind == 'overtake':
            e = {'type':'overtake','car_number':int(r['car_number']),
                 'participant': str(r['participant']) if pd.notna(r['participant']) else None,
                 'position_change': int(r['position_change'])}
        elif kind == 'race_control':
            e = {'type':'race_control','category':r['category'],
                 'text':r['message_text'],
                 'attrs': json.loads(r['attrs_json']) if pd.notna(r['attrs_json']) and r['attrs_json'] else {}}
        elif kind == 'lap_completed':
            e = {'type':'lap_completed','car_number':int(r['car_number']),
                 'lap_number':int(r['lap_num']),
                 'lap_time_s': round(float(r['lap_duration_s']),3),
                 'top_speed_kmh': int(r['top_speed']) if pd.notna(r['top_speed']) else None}
        out.setdefault(t_int, []).append(e)
    return out

events_by_tick = {}
for df, kind in [(am_activate_events,'attack_mode_activated'),
                 (am_deactivate_events,'attack_mode_deactivated'),
                 (ot_events,'overtake'),
                 (rc_events,'race_control'),
                 (lap_complete_events,'lap_completed')]:
    for t_int, evs in to_event_dicts(df, kind).items():
        events_by_tick.setdefault(t_int, []).extend(evs)

print(f"\nEvent ticks with at least one event: {len(events_by_tick)}")
print(f"Sample tick t=0 events: {events_by_tick.get(0, [])[:3]}")

# ============================================================
# Per-car AM budget at each tick
# Pre-compute: at each second, for each car, how many activations used and how many sec of budget left
# ============================================================
am_state = {}  # car -> list of (t_s, activations_used, remaining_budget_s)
TOTAL_BUDGET = 240.0
for car in sorted(drivers['car_number'].unique().tolist()):
    car_ams = am_windows[am_windows['car_number']==car].sort_values('start_s')
    states = []
    used = 0
    consumed = 0.0
    # State before first activation
    states.append((-100000, 0, TOTAL_BUDGET))
    for _, w in car_ams.iterrows():
        # At start of activation: budget unchanged until it ends
        # At end of activation: budget reduced by duration
        consumed += w['duration_s']
        used += 1
        states.append((w['end_s'], used, TOTAL_BUDGET - consumed))
    am_state[car] = states

def get_am_state(car, t):
    states = am_state[car]
    used, remaining = 0, TOTAL_BUDGET
    for ts, u, r in states:
        if ts <= t:
            used, remaining = u, r
        else:
            break
    return used, remaining

# ============================================================
# Build per-tick frames
# ============================================================
# Define race window for publication
PUBLISH_START_TICK = -10  # 10 seconds of pre-race
PUBLISH_END_TICK = int(RACE_DURATION_S) + 5

# Group enriched by race_time_s for fast access
enriched_indexed = enriched_v2.set_index(['race_time_s','car_number'])

frames = []
all_cars = sorted(drivers['car_number'].unique().tolist())

for tick in range(PUBLISH_START_TICK, PUBLISH_END_TICK + 1):
    cars_payload = []
    leader_lap = 0
    for car in all_cars:
        # Skip cars beyond their last data point
        try:
            row = enriched_indexed.loc[(float(tick), car)]
        except KeyError:
            continue

        is_retired = False
        if last_lap_end_time.get(car, RACE_DURATION_S) < tick - 30:  # 30s grace
            is_retired = True

        am_used, am_remaining = get_am_state(car, float(tick))

        lap_num = int(row['lap_num']) if row['lap_num'] > 0 else 0
        if lap_num > leader_lap and not is_retired:
            leader_lap = lap_num

        car_payload = {
            'car_number': car,
            'driver_short_name': driver_names.get(car, f'C{car}'),
            'position': int(row['resolved_position']),
            'current_lap': lap_num,
            'speed_kmh': round(float(row['speed_kmh']), 2),
            'gps': {
                'lat': round(float(row['gps_lat']), 6),
                'lng': round(float(row['gps_lng']), 6),
                'heading': round(float(row['gps_heading']), 2),
            },
            'accel_x': round(float(row['accel_x']), 3),
            'accel_y': round(float(row['accel_y']), 3),
            'brake_pct': round(float(row['brake_pct']), 1),
            'steer': round(float(row['steer']), 1),
            'yaw_rate': round(float(row['yaw_rate']), 3),
            'energy': {
                'pct_remaining': round(float(row['energy_pct_remaining']), 3),
                'kwh_remaining': round(float(row['energy_pct_remaining']) * 41.0 / 100.0, 3),
                'pct_used': round(float(row['energy_pct_used']), 3),
            },
            'attack_mode': {
                'active': bool(row['am_active']),
                'activations_used': int(am_used),
                'scenario': int(row['scenario']),
                'remaining_budget_s': round(float(am_remaining), 1),
            },
            'is_retired': is_retired,
        }
        cars_payload.append(car_payload)

    # Race phase from the first car (it's a global value)
    phase = 'pre_race' if tick < 0 else 'racing'
    if cars_payload:
        # Pull phase from any row at this tick
        any_row_phase = enriched_v2[enriched_v2['race_time_s']==float(tick)]['phase'].iloc[0] if (enriched_v2['race_time_s']==float(tick)).any() else phase
        phase = any_row_phase

    frame = {
        'schema_version': '1.0',
        'race_id': 'berlin_2024_r10',
        'race_time_s': tick,
        'race_duration_s': round(RACE_DURATION_S, 2),
        'pct_complete': round(max(0, min(100, tick / RACE_DURATION_S * 100)), 2),
        'race_phase': phase,
        'current_leader_lap': leader_lap,
        'cars': cars_payload,
        'events': events_by_tick.get(tick, []),
    }
    frames.append(frame)

print(f"\nBuilt {len(frames)} frames")
print(f"First frame summary: t={frames[0]['race_time_s']}, cars={len(frames[0]['cars'])}, events={len(frames[0]['events'])}")
print(f"Last frame summary:  t={frames[-1]['race_time_s']}, cars={len(frames[-1]['cars'])}, events={len(frames[-1]['events'])}")

# Spot-check a few frames
import json
print(f"\n=== Sample frame at t=0 (green flag) ===")
sample = frames[10]  # t=0 since we start at -10
print(f"phase={sample['race_phase']}, leader_lap={sample['current_leader_lap']}, cars={len(sample['cars'])}, events={len(sample['events'])}")
print(f"First car payload:\n{json.dumps(sample['cars'][0], indent=2)}")
if sample['events']:
    print(f"First event: {json.dumps(sample['events'][0], indent=2)}")

# ============================================================
# Write to local file then upload to GCS
# ============================================================
local_path = '/tmp/frames_v1.jsonl.gz'
print(f"\nWriting to {local_path}...")
with gzip.open(local_path, 'wt') as f:
    for frame in frames:
        f.write(json.dumps(frame) + '\n')

import os
size_mb = os.path.getsize(local_path) / 1e6
print(f"File size: {size_mb:.2f} MB")

# Upload to GCS
from google.cloud import storage
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket('class-demo')
blob = bucket.blob('formula-e/r10/simulator/frames_v1.jsonl.gz')
blob.upload_from_filename(local_path)
print(f"Uploaded to gs://class-demo/formula-e/r10/simulator/frames_v1.jsonl.gz")
print(f"Public URI: https://storage.googleapis.com/class-demo/formula-e/r10/simulator/frames_v1.jsonl.gz")

Retirement final laps: {1: 41, 2: 41, 3: 41, 4: 41, 5: 41, 7: 10, 8: 41, 9: 41, 11: 41, 13: 41, 16: 41, 17: 41, 18: 41, 21: 41, 22: 41, 23: 24, 25: 41, 33: 41, 37: 41, 48: 41, 51: 41, 94: 41}

Event counts: am_activate=44, am_deactivate=44, overtake=1248, race_control=58, lap_complete=852

Event ticks with at least one event: 914
Sample tick t=0 events: [{'type': 'race_control', 'category': 'flag.green', 'text': 'GREEN FLAG', 'attrs': {}}]

Built 2883 frames
First frame summary: t=-10, cars=22, events=0
Last frame summary:  t=2872, cars=20, events=0

=== Sample frame at t=0 (green flag) ===
phase=racing, leader_lap=1, cars=22, events=1
First car payload:
{
  "car_number": 1,
  "driver_short_name": "DEN",
  "position": 1,
  "current_lap": 1,
  "speed_kmh": 52.18,
  "gps": {
    "lat": 52.480181,
    "lng": 13.39322,
    "heading": 47.73
  },
  "accel_x": 0.016,
  "accel_y": 0.586,
  "brake_pct": 0.0,
  "steer": 2185.9,
  "yaw_rate": 0.916,
  "energy": {
    "pct_remaining": 99.914,
    

In [ ]:
# Cell 9b — Quick sanity check on the produced frames
import gzip, json

frames_loaded = []
with gzip.open('/tmp/frames_v1.jsonl.gz', 'rt') as f:
    for line in f:
        frames_loaded.append(json.loads(line))

print(f"Loaded {len(frames_loaded)} frames")
print(f"Tick range: {frames_loaded[0]['race_time_s']} to {frames_loaded[-1]['race_time_s']}")

# Pre-race sanity
print("\n=== Pre-race frame at t=-10 (car 1 = pole sitter) ===")
f10 = frames_loaded[0]
c1 = next(c for c in f10['cars'] if c['car_number']==1)
print(f"phase={f10['race_phase']}, leader_lap={f10['current_leader_lap']}, events={len(f10['events'])}")
print(f"Car 1: pos={c1['position']}, lap={c1['current_lap']}, speed={c1['speed_kmh']}, energy={c1['energy']['pct_remaining']}")

# Green flag tick
print(f"\n=== Green flag at t=0 ===")
f0 = frames_loaded[10]
c1 = next(c for c in f0['cars'] if c['car_number']==1)
c13 = next(c for c in f0['cars'] if c['car_number']==13)
print(f"phase={f0['race_phase']}, leader_lap={f0['current_leader_lap']}, events={[e['type'] for e in f0['events']]}")
print(f"Car 1:  pos={c1['position']}, speed={c1['speed_kmh']}, energy={c1['energy']['pct_remaining']}")
print(f"Car 13: pos={c13['position']}, speed={c13['speed_kmh']}, energy={c13['energy']['pct_remaining']}, scenario={c13['attack_mode']['scenario']}")

# Race-time 200 (mid-stint, around first AM activation window)
print(f"\n=== t=200 (early racing, AM windows opening) ===")
f200 = frames_loaded[210]
events_summary = {}
for e in f200['events']:
    events_summary[e['type']] = events_summary.get(e['type'], 0) + 1
print(f"phase={f200['race_phase']}, leader_lap={f200['current_leader_lap']}, events_by_type={events_summary}")
am_active_cars = [c['car_number'] for c in f200['cars'] if c['attack_mode']['active']]
print(f"Cars currently with AM active: {am_active_cars}")

# Race time 1500 (mid-race)
print(f"\n=== t=1500 (mid-race) ===")
f1500 = frames_loaded[1510]
c13 = next(c for c in f1500['cars'] if c['car_number']==13)
print(f"phase={f1500['race_phase']}, leader_lap={f1500['current_leader_lap']}")
print(f"Car 13: pos={c13['position']}, lap={c13['current_lap']}, energy={c13['energy']['pct_remaining']}, AM_used={c13['attack_mode']['activations_used']}, AM_budget={c13['attack_mode']['remaining_budget_s']}")

# Chequered flag
print(f"\n=== Around chequered flag ===")
cheq_idx = next(i for i,f in enumerate(frames_loaded) if any(e.get('category')=='flag.chequered' for e in f['events']))
print(f"Chequered flag found at frame index {cheq_idx}, race_time_s={frames_loaded[cheq_idx]['race_time_s']}")
fc = frames_loaded[cheq_idx]
print(f"phase={fc['race_phase']}, events={[e.get('type')+':'+e.get('category','') for e in fc['events']]}")

# Retirement check
print(f"\n=== Car 7 (retired lap 10) status across the race ===")
for t in [0, 500, 994, 1000, 2000, 2800]:
    if t+10 < len(frames_loaded):
        f = frames_loaded[t+10]
        c7 = next((c for c in f['cars'] if c['car_number']==7), None)
        if c7:
            print(f"  t={t}: pos={c7['position']}, lap={c7['current_lap']}, is_retired={c7['is_retired']}, speed={c7['speed_kmh']}")
        else:
            print(f"  t={t}: car 7 not in frame")

# Event distribution by type across the race
print(f"\n=== Event type totals across all frames ===")
from collections import Counter
all_events = Counter()
for f in frames_loaded:
    for e in f['events']:
        all_events[e['type']] += 1
print(dict(all_events))

# File size summary
import os
print(f"\nFile: {os.path.getsize('/tmp/frames_v1.jsonl.gz')/1e6:.2f} MB compressed")
print(f"Uncompressed bytes estimate: {sum(len(json.dumps(f)) for f in frames_loaded[:100])/100 * len(frames_loaded)/1e6:.1f} MB")

Loaded 2883 frames
Tick range: -10 to 2872

=== Pre-race frame at t=-10 (car 1 = pole sitter) ===
phase=pre_race, leader_lap=0, events=0
Car 1: pos=1, lap=0, speed=0.0, energy=100.0

=== Green flag at t=0 ===
phase=racing, leader_lap=1, events=['race_control']
Car 1:  pos=1, speed=52.18, energy=99.914
Car 13: pos=10, speed=69.79, energy=99.816, scenario=2

=== t=200 (early racing, AM windows opening) ===
phase=racing, leader_lap=3, events_by_type={}
Cars currently with AM active: [3, 4, 5, 8, 11, 16, 18, 21, 22, 23, 33, 37, 51]

=== t=1500 (mid-race) ===
phase=safety_car, leader_lap=22
Car 13: pos=1, lap=22, energy=49.694, AM_used=2, AM_budget=-0.1

=== Around chequered flag ===
Chequered flag found at frame index 2877, race_time_s=2867
phase=racing, events=['race_control:flag.chequered']

=== Car 7 (retired lap 10) status across the race ===
  t=0: pos=5, lap=1, is_retired=False, speed=50.83
  t=500: pos=14, lap=8, is_retired=False, speed=178.13
  t=994: pos=9, lap=0, is_retired=True,

In [ ]:
# Cell 10 — Diagnose the five issues
import pandas as pd
import json, gzip

print("=== Issue 1: AM budget going negative ===")
# Check what the per-car AM total actually is
am_totals = am_windows.groupby('car_number')['duration_s'].sum()
print(f"AM totals per car (max overflow above 240.0):")
print((am_totals - 240.0).describe())
print(f"Max overflow: {(am_totals - 240.0).max():.4f}s")
print(f"Cars with overflow > 0.05s: {((am_totals - 240.0) > 0.05).sum()}")

print(f"\n=== Issue 2: Missing lap_completed events ===")
# Check for lap_end_s values colliding on the same integer second
lap_complete = lap_windows_clean[['car_number','lap_num','lap_end_s']].copy()
lap_complete['t_int'] = lap_complete['lap_end_s'].astype(int)
collisions = lap_complete.groupby(['car_number','t_int']).size()
print(f"Same car, same second collisions: {(collisions>1).sum()} (would lose {(collisions-1).clip(lower=0).sum()} events)")
# Different cars, same second
multi_car = lap_complete.groupby('t_int').size()
print(f"Seconds with multiple lap completions across cars: {(multi_car>1).sum()}")
print(f"Total events expected after groupby key: {len(lap_complete)}")

# Actually, let's check: how many ticks in events_by_tick contain lap_completed entries?
import gzip, json
frames_loaded = []
with gzip.open('/tmp/frames_v1.jsonl.gz', 'rt') as f:
    for line in f:
        frames_loaded.append(json.loads(line))
lap_events_in_frames = sum(1 for f in frames_loaded for e in f['events'] if e['type']=='lap_completed')
print(f"lap_completed events in frames: {lap_events_in_frames}, expected: {len(lap_complete)}")

print(f"\n=== Issue 3: 13 cars marked AM-active at t=200 ===")
# Look at the actual AM windows that contain t=200
am_at_200 = am_windows[(am_windows['start_s'] <= 200) & (am_windows['end_s'] > 200)]
print(f"AM windows actually active at race_time=200:\n{am_at_200.to_string(index=False)}")
print(f"\nCars marked am_active=True at t=200 in enriched_v2:")
print(enriched_v2[(enriched_v2['race_time_s']==200) & (enriched_v2['am_active']==True)][['car_number','am_active','scenario']].to_string(index=False))

print(f"\n=== Issue 4: Chequered flag phase transition ===")
# Look at frames around t=2867
for i in range(2875, 2883):
    f = frames_loaded[i]
    has_cheq = any(e.get('category')=='flag.chequered' for e in f['events'])
    print(f"  frame[{i}]: t={f['race_time_s']}, phase={f['race_phase']}, has_chequered_event={has_cheq}")

print(f"\n=== Issue 5: Car 7 lap=0 at t=994 ===")
# What does enriched_v2 say about car 7 right before retirement?
c7 = enriched_v2[(enriched_v2['car_number']==7) & (enriched_v2['race_time_s'].between(980, 1000))]
print(c7[['race_time_s','lap_num','speed_kmh','energy_pct_remaining']].to_string(index=False))
print(f"\nCar 7 final_lap from energy: {final_laps[7]}")
print(f"Car 7 last lap_end_s from lap_windows: {last_lap_end_time[7]}")

=== Issue 1: AM budget going negative ===
AM totals per car (max overflow above 240.0):
count    22.000000
mean      0.020773
std       0.030341
min      -0.025000
25%      -0.002500
50%       0.017000
75%       0.041000
max       0.083000
Name: duration_s, dtype: float64
Max overflow: 0.0830s
Cars with overflow > 0.05s: 5

=== Issue 2: Missing lap_completed events ===
Same car, same second collisions: 0 (would lose 0 events)
Seconds with multiple lap completions across cars: 242
Total events expected after groupby key: 852
lap_completed events in frames: 832, expected: 852

=== Issue 3: 13 cars marked AM-active at t=200 ===
AM windows actually active at race_time=200:
 car_number  activation_num                        start_utc                          end_utc  start_s   end_s  duration_s
          3               1 2024-05-12 13:06:42.744000+00:00 2024-05-12 13:07:42.772000+00:00  157.018 217.046      60.028
          4               1 2024-05-12 13:06:38.846000+00:00 2024-05-12 13:0

In [ ]:
# Cell 11 — Rebuild frames with all five fixes applied
import pandas as pd
import numpy as np
import json, gzip, os
from collections import Counter

# --- Fix 1: Floor AM totals at exactly 240.0s for the budget tracker ---
# Recompute am_state with clipped totals
TOTAL_BUDGET = 240.0
am_state = {}
for car in sorted(drivers['car_number'].unique().tolist()):
    car_ams = am_windows[am_windows['car_number']==car].sort_values('start_s')
    # Floor each duration so the cumulative never exceeds 240.0
    total_so_far = 0.0
    states = [(-100000.0, 0, TOTAL_BUDGET)]
    used = 0
    for _, w in car_ams.iterrows():
        d = float(w['duration_s'])
        # Clip so we don't blow past 240 cumulatively
        d_effective = min(d, TOTAL_BUDGET - total_so_far)
        total_so_far += d_effective
        used += 1
        remaining = max(0.0, TOTAL_BUDGET - total_so_far)
        states.append((float(w['end_s']), used, remaining))
    am_state[car] = states

def get_am_state(car, t):
    states = am_state[car]
    used, remaining = 0, TOTAL_BUDGET
    for ts, u, r in states:
        if ts <= t:
            used, remaining = u, r
        else:
            break
    return used, remaining

# --- Fix 5: Per-car last-known-lap for retirement freeze ---
# For each car, build a lookup: at any t, what's the last lap they were genuinely in?
last_lap_seen = {}  # car -> dict mapping t_int -> lap_num
final_laps = energy.groupby('car_number')['lap_number'].max().to_dict()
last_lap_end_s_by_car = lap_windows_clean.groupby('car_number')['lap_end_s'].max().to_dict()

# --- Fix 2: Extend publish window to include events past chequered ---
PUBLISH_START_TICK = -10
PUBLISH_END_TICK = max(int(RACE_DURATION_S) + 5,
                       int(lap_windows_clean['lap_end_s'].max()) + 2)
print(f"Publish window: [{PUBLISH_START_TICK}, {PUBLISH_END_TICK}]")

# --- Rebuild events_by_tick ---
events_by_tick = {}

# AM activations
for _, r in am_windows.iterrows():
    t = int(r['start_s'])
    events_by_tick.setdefault(t, []).append({
        'type':'attack_mode_activated',
        'car_number':int(r['car_number']),
        'duration_s':round(float(r['duration_s']),1),
        'activation_num':int(r['activation_num']),
    })
    t = int(r['end_s'])
    events_by_tick.setdefault(t, []).append({
        'type':'attack_mode_deactivated',
        'car_number':int(r['car_number']),
        'activation_num':int(r['activation_num']),
    })

# Overtakes
for _, r in ot_events.iterrows():
    events_by_tick.setdefault(int(r['t_int']), []).append({
        'type':'overtake',
        'car_number':int(r['car_number']),
        'participant':str(r['participant']) if pd.notna(r['participant']) else None,
        'position_change':int(r['position_change']),
    })

# Race control
for _, r in rc_events.iterrows():
    events_by_tick.setdefault(int(r['t_int']), []).append({
        'type':'race_control',
        'category':r['category'],
        'text':r['message_text'],
        'attrs':json.loads(r['attrs_json']) if pd.notna(r['attrs_json']) and r['attrs_json'] else {},
    })

# Lap completions
for _, r in lap_complete_events.iterrows():
    events_by_tick.setdefault(int(r['t_int']), []).append({
        'type':'lap_completed',
        'car_number':int(r['car_number']),
        'lap_number':int(r['lap_num']),
        'lap_time_s':round(float(r['lap_duration_s']),3),
        'top_speed_kmh':int(r['top_speed']) if pd.notna(r['top_speed']) else None,
    })

print(f"Event ticks: {len(events_by_tick)}")

# --- Pre-compute last-known lap per car per second (forward-fill in-lap values) ---
# For each car: take rows where lap_num > 0, build a t→lap lookup, fill forward
enriched_v2 = enriched_v2.sort_values(['car_number','race_time_s']).reset_index(drop=True)
enriched_v2['lap_num_filled'] = enriched_v2['lap_num']
mask_invalid = enriched_v2['lap_num_filled'] <= 0
# Forward-fill within each car
enriched_v2.loc[mask_invalid, 'lap_num_filled'] = np.nan
enriched_v2['lap_num_filled'] = enriched_v2.groupby('car_number')['lap_num_filled'].ffill()
# For pre-race (before any lap started) fill with 0
enriched_v2['lap_num_filled'] = enriched_v2['lap_num_filled'].fillna(0).astype(int)

# Sanity check car 7
print(f"\nCar 7 lap_num_filled around retirement:")
print(enriched_v2[(enriched_v2['car_number']==7) & (enriched_v2['race_time_s'].between(670, 700))]
      [['race_time_s','lap_num','lap_num_filled','speed_kmh']].to_string(index=False))

# --- Index for fast lookup ---
enriched_indexed = enriched_v2.set_index(['race_time_s','car_number'])
all_cars = sorted(drivers['car_number'].unique().tolist())

# --- Rebuild frames ---
frames = []
RETIREMENT_GRACE_S = 30

for tick in range(PUBLISH_START_TICK, PUBLISH_END_TICK + 1):
    cars_payload = []
    leader_lap = 0
    has_chequered_event = any(e.get('category')=='flag.chequered' for e in events_by_tick.get(tick, []))

    for car in all_cars:
        try:
            row = enriched_indexed.loc[(float(tick), car)]
        except KeyError:
            continue

        # Retirement detection
        last_lap_t = last_lap_end_s_by_car.get(car, RACE_DURATION_S)
        is_retired = tick > last_lap_t + RETIREMENT_GRACE_S

        am_used, am_remaining = get_am_state(car, float(tick))

        # Use lap_num_filled so retired cars freeze on their last lap
        lap_num = int(row['lap_num_filled'])
        if lap_num > leader_lap and not is_retired:
            leader_lap = lap_num

        cars_payload.append({
            'car_number': car,
            'driver_short_name': driver_names.get(car, f'C{car}'),
            'position': int(row['resolved_position']),
            'current_lap': lap_num,
            'speed_kmh': round(float(row['speed_kmh']), 2),
            'gps': {
                'lat': round(float(row['gps_lat']), 6),
                'lng': round(float(row['gps_lng']), 6),
                'heading': round(float(row['gps_heading']), 2),
            },
            'accel_x': round(float(row['accel_x']), 3),
            'accel_y': round(float(row['accel_y']), 3),
            'brake_pct': round(float(row['brake_pct']), 1),
            'steer': round(float(row['steer']), 1),
            'yaw_rate': round(float(row['yaw_rate']), 3),
            'energy': {
                'pct_remaining': round(float(row['energy_pct_remaining']), 3),
                'kwh_remaining': round(float(row['energy_pct_remaining']) * 41.0 / 100.0, 3),
                'pct_used': round(float(row['energy_pct_used']), 3),
            },
            'attack_mode': {
                'active': bool(row['am_active']),
                'activations_used': int(am_used),
                'scenario': int(row['scenario']),
                'remaining_budget_s': round(max(0.0, float(am_remaining)), 1),   # Fix 1
            },
            'is_retired': is_retired,
        })

    # Determine phase. Backward merge_asof gave us the value. Override to chequered when the event fires this tick. (Fix 4)
    if cars_payload:
        phase_row = enriched_v2[enriched_v2['race_time_s']==float(tick)]
        phase = phase_row['phase'].iloc[0] if len(phase_row) else ('pre_race' if tick < 0 else 'racing')
    else:
        phase = 'pre_race' if tick < 0 else 'chequered'
    if has_chequered_event:
        phase = 'chequered'

    frames.append({
        'schema_version':'1.0',
        'race_id':'berlin_2024_r10',
        'race_time_s':tick,
        'race_duration_s':round(RACE_DURATION_S, 2),
        'pct_complete':round(max(0, min(100, tick / RACE_DURATION_S * 100)), 2),
        'race_phase':phase,
        'current_leader_lap':leader_lap,
        'cars':cars_payload,
        'events':events_by_tick.get(tick, []),
    })

print(f"\nBuilt {len(frames)} frames")

# --- Validate the fixes ---
print(f"\n=== Validation ===")
event_counts = Counter()
for f in frames:
    for e in f['events']:
        event_counts[e['type']] += 1
print(f"Event totals: {dict(event_counts)}")
print(f"Expected lap_completed: {len(lap_complete_events)}, got: {event_counts['lap_completed']}")

# Issue 1: AM budget never negative
min_budget = min(c['attack_mode']['remaining_budget_s'] for f in frames for c in f['cars'])
print(f"Min AM remaining_budget_s across all frames: {min_budget} (should be 0.0 or positive)")

# Issue 4: Chequered frames
print(f"\nFrames around chequered flag:")
cheq_idx = next(i for i,f in enumerate(frames) if any(e.get('category')=='flag.chequered' for e in f['events']))
for i in range(max(0,cheq_idx-2), min(len(frames),cheq_idx+4)):
    print(f"  t={frames[i]['race_time_s']:5d}: phase={frames[i]['race_phase']:12s} chequered_event={any(e.get('category')=='flag.chequered' for e in frames[i]['events'])}")

# Issue 5: Car 7 retirement
print(f"\nCar 7 retirement status:")
for t in [500, 670, 700, 800, 994, 1500, 2800]:
    idx = t - PUBLISH_START_TICK
    if 0 <= idx < len(frames):
        c7 = next((c for c in frames[idx]['cars'] if c['car_number']==7), None)
        if c7:
            print(f"  t={t}: lap={c7['current_lap']}, is_retired={c7['is_retired']}, speed={c7['speed_kmh']}, energy={c7['energy']['pct_remaining']}")
        else:
            print(f"  t={t}: not in frame")

# --- Re-upload ---
local_path = '/tmp/frames_v1.jsonl.gz'
with gzip.open(local_path, 'wt') as f:
    for fr in frames:
        f.write(json.dumps(fr) + '\n')
size_mb = os.path.getsize(local_path) / 1e6
print(f"\nFile size: {size_mb:.2f} MB")

from google.cloud import storage
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket('class-demo')
blob = bucket.blob('formula-e/r10/simulator/frames_v1.jsonl.gz')
blob.upload_from_filename(local_path)
print(f"Uploaded to gs://class-demo/formula-e/r10/simulator/frames_v1.jsonl.gz")

Publish window: [-10, 2907]
Event ticks: 914

Car 7 lap_num_filled around retirement:
 race_time_s  lap_num  lap_num_filled  speed_kmh
       670.0       10              10     176.46
       671.0       10              10     192.68
       672.0       10              10     207.58
       673.0       10              10     220.08
       674.0       -1              10     218.85
       675.0       -1              10     212.35
       676.0       -1              10     206.33
       677.0       -1              10     184.71
       678.0       -1              10     149.57
       679.0       -1              10     119.25
       680.0       -1              10      87.14
       681.0       -1              10      64.87
       682.0       -1              10      44.12
       683.0       -1              10      41.88
       684.0       -1              10      27.70
       685.0       -1              10      14.94
       686.0       -1              10       4.15
       687.0       -1           

In [ ]:
# Cell 12 — Finish validation and upload (typo fix)
import json, gzip, os

print("Car count distribution across frames: ", end='')
from collections import Counter
car_counts = Counter(len(f['cars']) for f in frames)
print(dict(car_counts))

print(f"\nCar 7 (retired lap 10):")
for t in [500, 670, 700, 800, 994, 1500, 2800]:
    idx = t - PUBLISH_START_TICK
    if 0 <= idx < len(frames):
        c7 = next((c for c in frames[idx]['cars'] if c['car_number']==7), None)
        if c7:
            print(f"  t={t:4d}: lap={c7['current_lap']:2d}, pos={c7['position']:2d}, is_retired={c7['is_retired']}, "
                  f"speed={c7['speed_kmh']:6.2f}, energy={c7['energy']['pct_remaining']:5.1f}")
        else:
            print(f"  t={t}: not in frame")

print(f"\nCar 23 (retired lap 24):")
for t in [500, 1500, 2688, 2700, 2800]:
    idx = t - PUBLISH_START_TICK
    if 0 <= idx < len(frames):
        c23 = next((c for c in frames[idx]['cars'] if c['car_number']==23), None)
        if c23:
            print(f"  t={t:4d}: lap={c23['current_lap']:2d}, pos={c23['position']:2d}, is_retired={c23['is_retired']}, "
                  f"speed={c23['speed_kmh']:6.2f}, energy={c23['energy']['pct_remaining']:5.1f}")

# Find the one 21-car frame
print(f"\nFrame(s) with 21 cars instead of 22:")
for i, f in enumerate(frames):
    if len(f['cars']) == 21:
        car_nums = {c['car_number'] for c in f['cars']}
        missing = set(all_cars) - car_nums
        print(f"  frame[{i}] t={f['race_time_s']}: missing car(s) {missing}")

# Upload
local_path = '/tmp/frames_v1.jsonl.gz'
with gzip.open(local_path, 'wt') as f:
    for fr in frames:
        f.write(json.dumps(fr) + '\n')
print(f"\nFile size: {os.path.getsize(local_path)/1e6:.2f} MB")

from google.cloud import storage
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket('class-demo')
blob = bucket.blob('formula-e/r10/simulator/frames_v1.jsonl.gz')
blob.upload_from_filename(local_path)
print(f"Uploaded to gs://class-demo/formula-e/r10/simulator/frames_v1.jsonl.gz")

Car count distribution across frames: {22: 1004, 21: 1695, 20: 219}

Car 7 (retired lap 10):
  t= 500: lap= 8, pos=14, is_retired=False, speed=178.13, energy= 81.2
  t= 670: lap=10, pos=14, is_retired=False, speed=176.46, energy= 74.7
  t= 700: lap=10, pos= 9, is_retired=False, speed=  0.00, energy= 74.0
  t= 800: lap=10, pos= 9, is_retired=True, speed= 77.71, energy= 74.0
  t= 994: lap=10, pos= 9, is_retired=True, speed=  0.00, energy= 74.0
  t=1500: not in frame
  t=2800: not in frame

Car 23 (retired lap 24):
  t= 500: lap= 8, pos=15, is_retired=False, speed=187.46, energy= 81.1
  t=1500: lap=22, pos= 9, is_retired=False, speed=109.13, energy= 49.4
  t=2688: lap=24, pos= 7, is_retired=True, speed=  0.00, energy= 41.1

Frame(s) with 21 cars instead of 22:
  frame[507] t=497: missing car(s) {48}
  frame[1005] t=995: missing car(s) {7}
  frame[1006] t=996: missing car(s) {7}
  frame[1007] t=997: missing car(s) {7}
  frame[1008] t=998: missing car(s) {7}
  frame[1009] t=999: missing car

In [ ]:
# %%
# Cell 13 — frames_v2: real per-lap top speeds + overtake noise filter
#
# PASTE THIS WHOLE FILE as a new cell at the BOTTOM of build_frames.ipynb
# (after Cell 12b). It is NOT standalone — it uses the notebook namespace
# built by the earlier cells: frames, lap_windows_clean, drivers, R10,
# RACE_START_UTC, PROJECT_ID. Run all cells above it first.
#
# Two data-quality fixes, both found during Fork 2 live integration:
#
#   1. lap_completed.top_speed_kmh was sourced from laps.top_speed, which is
#      broken in the source data (always 0). Recover real values as
#      MAX(tv_speed) over the 20 Hz telemetry samples inside each lap window —
#      the same derivation as the fe_race10.top_speed_per_lap table in BQ.
#
#   2. Overtake events with position_change == 0 carry no information (they
#      are the no-op side of provisional position swaps) and flood the agent's
#      event queries — at one point ~14 of 50 recent events were zero-change
#      noise. Drop them from the event stream. The position-resolution logic
#      in Cell 8c is unaffected (a zero change was a no-op there too).
#
# Writes frames_v2.jsonl.gz ALONGSIDE v1 — v1 stays in place for diffing.
# schema_version bumps to 1.1; top_speed_kmh becomes a float (e.g. 222.1).

import pandas as pd
import numpy as np
import json, gzip, os
from collections import Counter

# %%
# Cell 13 §1 — Per-(car, lap) top speed from 20 Hz telemetry
print("Computing per-lap top speeds from telemetry (22 cars, ~1-2 min)...\n")

top_speed = {}  # (car_number, lap_num) -> km/h
for car in sorted(drivers['car_number'].unique().tolist()):
    tele = pd.read_parquet(f'{R10}/telemetry/car_number={car}/')
    tele['time'] = pd.to_datetime(tele['time']).dt.tz_localize('UTC')
    t_s = (tele['time'] - RACE_START_UTC).dt.total_seconds().values

    laps_c = lap_windows_clean[lap_windows_clean['car_number'] == car].sort_values('lap_num')
    starts = laps_c['lap_start_s'].values
    ends   = laps_c['lap_end_s'].values
    nums   = laps_c['lap_num'].values

    # Same bounded lap assignment as Cell 7: sample belongs to lap i iff
    # starts[i] <= t < ends[i]. Between-window samples are dropped.
    idx = np.searchsorted(starts, t_s, side='right') - 1
    idxc = np.clip(idx, 0, len(starts) - 1)
    in_lap = (idx >= 0) & (t_s >= starts[idxc]) & (t_s < ends[idxc])

    per_lap = (
        pd.DataFrame({'lap': nums[idxc[in_lap]],
                      'v': tele['tv_speed'].values[in_lap]})
        .groupby('lap')['v'].max()
    )
    for lap, vmax in per_lap.items():
        top_speed[(int(car), int(lap))] = round(float(vmax), 1)
    print(f"  car {car:3d}: {len(per_lap)} laps, max {per_lap.max():.1f} km/h")

print(f"\nComputed {len(top_speed)} (car, lap) top speeds")

# Anchor validation — DAC laps 1-5, known ground truth from BQ
# (fe_race10.top_speed_per_lap, verified during Fork 2 chunk 2.5)
EXPECTED_DAC = {1: 215.4, 2: 222.6, 3: 224.8, 4: 228.4, 5: 236.3}
print("\nAnchor check — car 13 (DAC) laps 1-5 vs BQ top_speed_per_lap:")
anchor_ok = True
for lap, expected in EXPECTED_DAC.items():
    got = top_speed.get((13, lap))
    delta = abs(got - expected) if got is not None else float('inf')
    flag = "✓" if delta <= 1.0 else "✗"
    if delta > 1.0:
        anchor_ok = False
    print(f"  {flag} lap {lap}: got {got}, expected {expected} (Δ{delta:.1f})")
assert anchor_ok, "Top-speed derivation disagrees with BQ ground truth — investigate before uploading"

# %%
# Cell 13 §2 — Patch the frames in memory
patched_laps = 0
missing_topspeed = 0
dropped_overtakes = 0
kept_overtakes = 0

for f in frames:
    f['schema_version'] = '1.1'
    new_events = []
    for e in f['events']:
        if e['type'] == 'overtake':
            if e.get('position_change') == 0:
                dropped_overtakes += 1
                continue
            kept_overtakes += 1
        elif e['type'] == 'lap_completed':
            ts = top_speed.get((e['car_number'], e['lap_number']))
            if ts is None:
                missing_topspeed += 1
            e = {**e, 'top_speed_kmh': ts}
            patched_laps += 1
        new_events.append(e)
    f['events'] = new_events

print("=== Patch summary ===")
print(f"lap_completed events patched: {patched_laps} "
      f"({missing_topspeed} with no telemetry coverage → None)")
print(f"overtake events: kept {kept_overtakes}, dropped {dropped_overtakes} zero-change")

# %%
# Cell 13 §3 — Validate the patched frames
event_counts = Counter()
zero_ts = 0
zero_change = 0
for f in frames:
    for e in f['events']:
        event_counts[e['type']] += 1
        if e['type'] == 'lap_completed' and e['top_speed_kmh'] in (0, 0.0):
            zero_ts += 1
        if e['type'] == 'overtake' and e.get('position_change') == 0:
            zero_change += 1

print("=== Validation ===")
print(f"Event totals: {dict(event_counts)}")
print(f"lap_completed with top_speed_kmh == 0: {zero_ts} (must be 0)")
print(f"overtake with position_change == 0:    {zero_change} (must be 0)")
assert zero_ts == 0, "zero top speeds remain"
assert zero_change == 0, "zero-change overtakes remain"

# Spot-check a mid-race frame's lap events
sample = next(f for f in frames
              if any(e['type'] == 'lap_completed' for e in f['events'])
              and f['race_time_s'] > 200)
print(f"\nSample frame t={sample['race_time_s']} (schema {sample['schema_version']}):")
for e in sample['events']:
    print(f"  {e}")

# %%
# Cell 13 §4 — Write + upload frames_v2 (v1 untouched)
local_path = '/tmp/frames_v2.jsonl.gz'
with gzip.open(local_path, 'wt') as fh:
    for fr in frames:
        fh.write(json.dumps(fr) + '\n')
size_mb = os.path.getsize(local_path) / 1e6
print(f"File size: {size_mb:.2f} MB")

from google.cloud import storage
gcs = storage.Client(project=PROJECT_ID)
blob = gcs.bucket('class-demo').blob('formula-e/r10/simulator/frames_v2.jsonl.gz')
blob.upload_from_filename(local_path)
print("Uploaded to gs://class-demo/formula-e/r10/simulator/frames_v2.jsonl.gz")
print("\nNext: update FRAMES_PATH defaults (config.py, deploy.sh) and update the running service.")

Computing per-lap top speeds from telemetry (22 cars, ~1-2 min)...

  car   1: 41 laps, max 243.9 km/h
  car   2: 41 laps, max 246.5 km/h
  car   3: 41 laps, max 244.8 km/h
  car   4: 41 laps, max 253.5 km/h
  car   5: 41 laps, max 245.5 km/h
  car   7: 10 laps, max 243.0 km/h
  car   8: 41 laps, max 244.3 km/h
  car   9: 41 laps, max 239.6 km/h
  car  11: 41 laps, max 243.5 km/h
  car  13: 41 laps, max 239.1 km/h
  car  16: 41 laps, max 247.7 km/h
  car  17: 39 laps, max 248.0 km/h
  car  18: 41 laps, max 237.8 km/h
  car  21: 41 laps, max 243.7 km/h
  car  22: 41 laps, max 245.2 km/h
  car  23: 24 laps, max 238.9 km/h
  car  25: 41 laps, max 253.7 km/h
  car  33: 41 laps, max 237.0 km/h
  car  37: 41 laps, max 238.6 km/h
  car  48: 41 laps, max 251.5 km/h
  car  51: 41 laps, max 244.6 km/h
  car  94: 41 laps, max 239.1 km/h

Computed 852 (car, lap) top speeds

Anchor check — car 13 (DAC) laps 1-5 vs BQ top_speed_per_lap:
  ✓ lap 1: got 215.4, expected 215.4 (Δ0.0)
  ✓ lap 2: got 222.

In [21]:
# %%
# Cell 14 — frames_v3: fix overtake identity (grid position → real car number)
#
# STANDALONE: does NOT need the pipeline cells. Paste as a new cell at the
# bottom of build_frames.ipynb (or run in a fresh kernel) — it reads
# frames_v2.jsonl.gz from GCS, patches it, writes frames_v3.jsonl.gz.
#
# The bug (verified by position-adjacency scoring, 2026-06-04): in the
# source timing data's overtake records, the "car_number" field is the
# subject car's GRID POSITION (domain 1-22), while "participant" is a real
# car number. frames_v1/v2 passed the grid ID through as car_number, so
# every overtake event misattributed its subject except where grid position
# coincidentally equals a real car number. Example: frame events showing
# "car 13" overtakes were actually grid-P13 = car 18 (Daruvala), not DAC.
#
# Positions in the cars[] payload are NOT affected (Cell 8c resolves them
# from lap-end anchors). Only events[] entries with type == 'overtake'
# change. schema_version bumps to 1.2.

import gzip
import io
import json
from collections import Counter

import pandas as pd
from google.cloud import storage

BUCKET = "class-demo"
V2_PATH = "formula-e/r10/simulator/frames_v2.jsonl.gz"
V3_PATH = "formula-e/r10/simulator/frames_v3.jsonl.gz"
GRID_PARQUET = f"gs://{BUCKET}/formula-e/berlin_2024/r10/timing/startgrid.parquet"

import subprocess
PROJECT_ID = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# %%
# Cell 14 §1 — Load the grid map and frames_v2
grid = pd.read_parquet(GRID_PARQUET)
grid_map = dict(zip(grid["position"].astype(int), grid["car_number"].astype(int)))
print(f"Grid map: {len(grid_map)} positions -> cars")
assert len(grid_map) == 22, "expected 22 grid positions"
entry_cars = set(grid_map.values())

gcs = storage.Client(project=PROJECT_ID)
blob = gcs.bucket(BUCKET).blob(V2_PATH)
raw = blob.download_as_bytes()
frames = [json.loads(line) for line in gzip.decompress(raw).decode().splitlines()]
print(f"Loaded {len(frames)} frames from gs://{BUCKET}/{V2_PATH}")
sv = {f["schema_version"] for f in frames}
assert sv == {"1.1"}, f"expected schema 1.1 input, got {sv} — wrong input file?"

# %%
# Cell 14 §2 — Remap overtake subjects through the grid map
remapped = 0
unmappable = 0
before_ids = Counter()
after_ids = Counter()

for f in frames:
    f["schema_version"] = "1.2"
    for e in f["events"]:
        if e["type"] != "overtake":
            continue
        raw_id = e["car_number"]
        before_ids[raw_id] += 1
        real = grid_map.get(raw_id)
        if real is None:
            # Should not happen: grid IDs are 1-22. Keep but flag loudly.
            unmappable += 1
            continue
        e["car_number"] = real
        after_ids[real] += 1
        remapped += 1

print(f"Remapped {remapped} overtake events; {unmappable} unmappable (must be 0)")
assert unmappable == 0, "overtake car_number outside grid domain 1-22 — investigate"

# %%
# Cell 14 §3 — Validate
# (a) After remap, every overtake subject must be a real entry-list car,
#     and high car numbers (23+) must now APPEAR (they were impossible before).
bad = [c for c in after_ids if c not in entry_cars]
assert not bad, f"non-entry-list cars after remap: {bad}"
high_cars = {c: n for c, n in after_ids.items() if c > 22}
print(f"(a) Overtake subjects now include {len(high_cars)} cars numbered >22: {high_cars}")
assert high_cars, "expected cars >22 to appear after remap"

# (b) DAC (car 13, started P10) should be among the most active —
#     the P10->P1 winner. Before the fix, raw ID 10 was a 'phantom'.
top5 = after_ids.most_common(5)
print(f"(b) Most overtake-involved subjects after remap: {top5}")
assert any(car == 13 for car, _ in top5), "expected DAC among top-5 most active"

# (c) Participants were always real car numbers — untouched. Spot-print one
#     event for eyeball sanity.
sample = next(e for f in frames for e in f["events"]
              if e["type"] == "overtake" and e.get("participant"))
print(f"(c) Sample remapped overtake event: {sample}")

# (d) Non-overtake events untouched: counts must match v2 totals exactly.
counts = Counter(e["type"] for f in frames for e in f["events"])
print(f"(d) Event totals: {dict(counts)}")

# %%
# Cell 14 §4 — Write + upload frames_v3 (v2 untouched)
buf = io.BytesIO()
with gzip.GzipFile(fileobj=buf, mode="wb") as gz:
    for fr in frames:
        gz.write((json.dumps(fr) + "\n").encode())
data = buf.getvalue()
print(f"File size: {len(data) / 1e6:.2f} MB")

gcs.bucket(BUCKET).blob(V3_PATH).upload_from_string(
    data, content_type="application/gzip"
)
print(f"Uploaded to gs://{BUCKET}/{V3_PATH}")
print("\nNext: flip FRAMES_PATH defaults to v3, update the running service,")
print("reset Firestore (stale grid-encoded events), restart the replay.")

Grid map: 22 positions -> cars
Loaded 2918 frames from gs://class-demo/formula-e/r10/simulator/frames_v2.jsonl.gz
Remapped 716 overtake events; 0 unmappable (must be 0)
(a) Overtake subjects now include 7 cars numbered >22: {37: 37, 94: 34, 48: 37, 25: 44, 23: 30, 51: 11, 33: 5}
(b) Most overtake-involved subjects after remap: [(13, 51), (9, 45), (4, 45), (25, 44), (17, 42)]
(c) Sample remapped overtake event: {'type': 'overtake', 'car_number': 1, 'participant': '37', 'position_change': -1}
(d) Event totals: {'race_control': 58, 'overtake': 716, 'lap_completed': 852, 'attack_mode_activated': 44, 'attack_mode_deactivated': 44}
File size: 2.64 MB
Uploaded to gs://class-demo/formula-e/r10/simulator/frames_v3.jsonl.gz

Next: flip FRAMES_PATH defaults to v3, update the running service,
reset Firestore (stale grid-encoded events), restart the replay.
